# Prescriptively Unsafe: A Reproducible Audit of When Accurate ICU Prediction Models Would Harm Patients If Used to Guide Treatment

## Notebook 1 of 3: Data Foundation

**Scope of this notebook.** Cohort construction, exclusion accounting, feature engineering, treatment extraction, outcome generation, quality control, missingness characterisation, and exploratory analysis for two independent critical-care databases: MIMIC-IV v2.2 (development and internal temporal validation) and the eICU Collaborative Research Database v2.0 (external validation). The two analytic tables produced here are consumed unchanged by Notebook 2 (prediction, causal analysis, prescriptive-safety audit) and Notebook 3 (statistics, figures, journal outputs).

### Design rationale (why the data are built this way)

The scientific question is prescriptive, not predictive. We ask when a well-calibrated risk model, if used to *guide* treatment, would change treatment in a way that harms patients, because treatment lies on the causal path to the outcome. Answering that question imposes a specific data geometry that ordinary prognostic pipelines do not, so we fix it here at the foundation:

1. **A single decision anchor (landmark).** Every stay is evaluated at a fixed time `t0 + PRED_HORIZON_H` after ICU admission (default 24 h). Features are drawn only from the pre-anchor window `[t0, t0 + 24h]`. This is the moment at which a deployed score would inform ongoing care. Anchoring removes the immortal-time and reverse-causation problems that arise when features and treatments are pooled across the whole stay.

2. **Treatments are measured strictly after the anchor.** Vasopressor use, invasive and non-invasive ventilation, and renal replacement therapy are recorded in the post-anchor window `(t0 + 24h, ICU discharge]`. These are the actions a risk score would plausibly move. Separating the treatment window from the feature window is what makes the later g-formula, IPW, and marginal structural model estimands well defined.

3. **Landmark eligibility.** A stay must be alive and in the ICU at the anchor plus a short observation buffer, so that a post-anchor decision window exists. This is a deliberate landmark restriction, reported transparently in the CONSORT accounting, not a silent survivorship filter.

4. **Two databases, one schema.** MIMIC-IV and eICU are harmonised into an identical column dictionary so that transportability and external prescriptive safety can be evaluated without ad hoc recoding downstream.

### Reproducibility contract

Deterministic seeds, pinned physiologic plausibility ranges, cached intermediate artefacts (Parquet), checkpoint-and-resume on every expensive extraction, structured logging, and a written artefact manifest. Nothing here requires a GPU. Every large table (chartevents, labevents, vitalPeriodic, lab) is read once, in chunks, filtered to the cohort and to a whitelist of concept identifiers, and never re-scanned.

### How to run

Run cells top to bottom. Expensive extractions detect their own cache and skip on re-run, so the notebook is safe to restart. Set `FORCE_RECOMPUTE = True` in the configuration cell to invalidate caches. Estimated first-pass runtime on a 32 GB CPU workstation is dominated by the two vitals tables; everything else is minutes.

> Editorial conventions in prose: British spelling, no em dashes, exact numerical anchoring, and the word audit in preference to framework where either would do.


In [1]:
# =============================================================================
# CONFIGURATION
# All tunable parameters live here. Nothing below this cell should hard-code a
# path, a threshold, or a concept identifier.
# =============================================================================
from pathlib import Path

class CFG:
    # --- Raw data roots (folder-inside-folder layout: <root>/<section>/<name>.csv/<name>.csv) ---
    MIMIC_ROOT = Path(r"C:\mimic-iv-2.2")
    EICU_ROOT  = Path(r"C:\Users\kruta\Downloads\eicu-collaborative-research-database-2.0")

    # --- Output tree (created relative to this notebook) ---
    OUTPUT_DIR = Path("./nb1_outputs_adm")   # separate tree; its fresh cache forces recompute
    CACHE_DIR  = OUTPUT_DIR / "cache"
    FIG_DIR    = OUTPUT_DIR / "figures"
    TABLE_DIR  = OUTPUT_DIR / "tables"

    # --- Landmark / windowing (hours) ---
    PRED_HORIZON_H   = 6     # earlier landmark, less pre-anchor treatment in the features
    POST_BUFFER_H    = 4      # minimum post-anchor observation required for eligibility
    FEATURE_WINDOW_H = 6     # MUST equal PRED_HORIZON_H

    # --- Cohort rules ---
    MIN_AGE          = 18
    FIRST_STAY_KEY   = "subject"   # 'subject' -> first ICU stay per patient; 'hadm' -> per hospital admission

    # --- Engineering ---
    RANDOM_SEED      = 20260707
    FORCE_RECOMPUTE  = False
    CHUNK_LARGE      = 5_000_000   # chartevents, labevents, vitalPeriodic
    CHUNK_MED        = 1_500_000   # inputevents, procedureevents, eICU lab

    # --- Figure export ---
    FIG_DPI          = 300

    @classmethod
    def anchor_h(cls):
        return cls.PRED_HORIZON_H

    @classmethod
    def min_los_h(cls):
        # a stay must reach the anchor plus a buffer to be eligible
        return cls.PRED_HORIZON_H + cls.POST_BUFFER_H

for d in (CFG.OUTPUT_DIR, CFG.CACHE_DIR, CFG.FIG_DIR, CFG.TABLE_DIR):
    d.mkdir(parents=True, exist_ok=True)

print("Output tree ready under:", CFG.OUTPUT_DIR.resolve())
print("Decision anchor: t0 +", CFG.PRED_HORIZON_H, "h  |  minimum eligible ICU LOS:", CFG.min_los_h(), "h")


Output tree ready under: C:\Users\<user>\Downloads\nb1_outputs_adm
Decision anchor: t0 + 6 h  |  minimum eligible ICU LOS: 10 h


In [2]:
# =============================================================================
# ENVIRONMENT, LOGGING, REPRODUCIBILITY, VERSION REPORTING
# =============================================================================
import os, sys, gc, json, time, logging, platform, warnings, random, hashlib
from datetime import datetime
from contextlib import contextmanager

import numpy as np
import pandas as pd

# Optional but recommended libraries; guarded so the notebook still runs without them.
try:
    from tqdm.auto import tqdm
    _HAS_TQDM = True
except Exception:
    _HAS_TQDM = False
    def tqdm(x=None, **k):
        return x

import matplotlib
matplotlib.use("Agg")  # deterministic, headless-safe rendering; figures are written to disk
import matplotlib.pyplot as plt
try:
    import seaborn as sns
    sns.set_theme(context="paper", style="whitegrid")
    _HAS_SNS = True
except Exception:
    _HAS_SNS = False

try:
    from scipy import stats as sstats
    _HAS_SCIPY = True
except Exception:
    _HAS_SCIPY = False

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)
pd.options.mode.chained_assignment = None

# ---- Deterministic seeds ----
os.environ["PYTHONHASHSEED"] = str(CFG.RANDOM_SEED)
random.seed(CFG.RANDOM_SEED)
np.random.seed(CFG.RANDOM_SEED)

# ---- Structured logging (console + file) ----
LOG_PATH = CFG.OUTPUT_DIR / "nb1_run.log"
logger = logging.getLogger("prescriptive_audit")
logger.setLevel(logging.INFO)
logger.handlers.clear()
_fmt = logging.Formatter("%(asctime)s | %(levelname)-7s | %(message)s", "%H:%M:%S")
_sh = logging.StreamHandler(sys.stdout); _sh.setFormatter(_fmt); logger.addHandler(_sh)
_fh = logging.FileHandler(LOG_PATH, mode="w", encoding="utf-8"); _fh.setFormatter(_fmt); logger.addHandler(_fh)
logger.propagate = False

@contextmanager
def timed(label):
    t = time.time()
    logger.info("START  %s", label)
    try:
        yield
    finally:
        logger.info("DONE   %s  (%.1fs)", label, time.time() - t)

def _v(mod):
    try:
        return __import__(mod).__version__
    except Exception:
        return "not installed"

logger.info("Python %s on %s", sys.version.split()[0], platform.platform())
for m in ("numpy", "pandas", "scipy", "matplotlib", "seaborn", "pyarrow", "sklearn"):
    logger.info("  %-11s %s", m, _v(m))
logger.info("Seed = %d | tqdm=%s | seaborn=%s | scipy=%s", CFG.RANDOM_SEED, _HAS_TQDM, _HAS_SNS, _HAS_SCIPY)

RUN_MANIFEST = {"started": datetime.now().isoformat(timespec="seconds"),
                "seed": CFG.RANDOM_SEED, "artefacts": {}}


C:\Users\<user>\anaconda3-v2\Lib\site-packages\pandas\core\arrays\masked.py:61: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


21:11:56 | INFO    | Python 3.11.5 on Windows-10-10.0.26200-SP0
21:11:56 | INFO    |   numpy       1.24.4
21:11:56 | INFO    |   pandas      2.3.3
21:11:56 | INFO    |   scipy       1.11.1
21:11:56 | INFO    |   matplotlib  3.7.2
21:11:56 | INFO    |   seaborn     0.12.2
21:11:56 | INFO    |   pyarrow     24.0.0
21:11:56 | INFO    |   sklearn     1.3.0
21:11:56 | INFO    | Seed = 20260707 | tqdm=True | seaborn=True | scipy=True


In [3]:
# =============================================================================
# FILESYSTEM RESOLVER  (folder-inside-folder layout)
# Each logical table <name> is a directory named "<name>.csv" that contains the
# actual data file. We resolve the inner file rather than assuming its extension,
# so uncompressed .csv and gzipped .csv.gz extracts both work unchanged.
# =============================================================================
def _resolve_inner(folder: Path) -> Path:
    """Return the data file inside a '<name>.csv' directory. Handles .csv and .csv.gz."""
    if folder.is_file():
        return folder  # tolerate a flat layout too
    if not folder.exists():
        raise FileNotFoundError(f"Missing table directory: {folder}")
    candidates = sorted([p for p in folder.iterdir()
                         if p.is_file() and p.suffix.lower() in (".csv", ".gz")])
    if not candidates:
        raise FileNotFoundError(f"No .csv/.csv.gz file inside: {folder}")
    # prefer an exact-name match, else the first csv/gz
    stem = folder.name
    for p in candidates:
        if p.name == stem or p.name == stem + ".gz":
            return p
    return candidates[0]

def mimic_path(section: str, name: str) -> Path:
    """section in {'hosp','icu'}; name e.g. 'admissions' (no extension)."""
    return _resolve_inner(CFG.MIMIC_ROOT / section / f"{name}.csv")

def eicu_path(name: str) -> Path:
    """name uses eICU's exact casing, e.g. 'vitalPeriodic', 'apachePatientResult'."""
    return _resolve_inner(CFG.EICU_ROOT / f"{name}.csv")

def _fsize_mb(p: Path) -> float:
    try:
        return p.stat().st_size / 1e6
    except Exception:
        return float("nan")

def validate_layout():
    """Probe the tables this notebook needs and report presence and size. Non-fatal."""
    needed_mimic = {"hosp": ["patients", "admissions", "diagnoses_icd", "labevents"],
                    "icu":  ["icustays", "chartevents", "inputevents", "procedureevents"]}
    needed_eicu  = ["patient", "lab", "vitalPeriodic", "vitalAperiodic",
                    "treatment", "infusionDrug", "apachePatientResult", "apacheApsVar", "diagnosis"]
    rows = []
    for section, names in needed_mimic.items():
        for n in names:
            try:
                p = mimic_path(section, n); rows.append(("MIMIC", f"{section}/{n}", "ok", f"{_fsize_mb(p):.0f} MB", str(p)))
            except Exception as e:
                rows.append(("MIMIC", f"{section}/{n}", "MISSING", "-", str(e)))
    for n in needed_eicu:
        try:
            p = eicu_path(n); rows.append(("eICU", n, "ok", f"{_fsize_mb(p):.0f} MB", str(p)))
        except Exception as e:
            rows.append(("eICU", n, "MISSING", "-", str(e)))
    rep = pd.DataFrame(rows, columns=["db", "table", "status", "size", "path"])
    miss = rep[rep.status != "ok"]
    if len(miss):
        logger.warning("Some tables were not found. Adjust CFG.MIMIC_ROOT / CFG.EICU_ROOT if needed:")
        for _, r in miss.iterrows():
            logger.warning("   %-5s %-18s -> %s", r.db, r.table, r.path)
    else:
        logger.info("All required tables located.")
    return rep

LAYOUT = validate_layout()
LAYOUT[["db", "table", "status", "size"]]


21:11:56 | INFO    | All required tables located.


,db,table,status,size
0,MIMIC,hosp/patients,ok,10 MB
1,MIMIC,hosp/admissions,ok,73 MB
2,MIMIC,hosp/diagnoses_icd,ok,135 MB
3,MIMIC,hosp/labevents,ok,13730 MB
4,MIMIC,icu/icustays,ok,11 MB
5,MIMIC,icu/chartevents,ok,30204 MB
6,MIMIC,icu/inputevents,ok,2346 MB
7,MIMIC,icu/procedureevents,ok,129 MB
8,eICU,patient,ok,48 MB
9,eICU,lab,ok,2364 MB


In [4]:
# =============================================================================
# CORE UTILITIES: caching, chunked event extraction, aggregation, figure export
# =============================================================================
def cache_path(name: str) -> Path:
    return CFG.CACHE_DIR / f"{name}.parquet"

def cache_exists(name: str) -> bool:
    return cache_path(name).exists() and not CFG.FORCE_RECOMPUTE

def save_cache(df: pd.DataFrame, name: str) -> Path:
    p = cache_path(name)
    df.to_parquet(p, index=False)
    RUN_MANIFEST["artefacts"][name] = {"path": str(p), "rows": int(len(df)), "cols": int(df.shape[1])}
    logger.info("cached %-28s rows=%-8d cols=%d", name, len(df), df.shape[1])
    return p

def load_cache(name: str) -> pd.DataFrame:
    return pd.read_parquet(cache_path(name))

def downcast(df: pd.DataFrame) -> pd.DataFrame:
    """Shrink numeric dtypes to conserve memory without changing values materially."""
    for c in df.select_dtypes(include=["float64"]).columns:
        df[c] = pd.to_numeric(df[c], downcast="float")
    for c in df.select_dtypes(include=["int64"]).columns:
        df[c] = pd.to_numeric(df[c], downcast="integer")
    return df

def savefig(fig, name: str):
    """Write a figure as both 300-dpi PNG and vector PDF."""
    png = CFG.FIG_DIR / f"{name}.png"
    pdf = CFG.FIG_DIR / f"{name}.pdf"
    fig.savefig(png, dpi=CFG.FIG_DPI, bbox_inches="tight")
    fig.savefig(pdf, bbox_inches="tight")
    plt.close(fig)
    logger.info("figure -> %s (.png/.pdf)", name)

def extract_events_windowed(inner_path, id_col, time_col, item_col, value_col,
                            itemids, windows, chunksize, dtype=None,
                            value_transforms=None, label="events"):
    """Stream a large event table once, keeping only rows whose concept id is in
    `itemids`, whose subject id is in `windows`, and whose timestamp falls inside
    that subject's [win_start, win_end].

    windows : DataFrame with columns [id_col, 'win_start', 'win_end'] (datetime).
    value_transforms : optional dict {itemid: callable(series)->series} applied to
                       value_col before aggregation (e.g. Fahrenheit to Celsius).
    Returns a long DataFrame [id_col, item_col, value_col, time_col].
    """
    ids = set(windows[id_col].tolist())
    itemset = set(itemids)
    wmap = windows.set_index(id_col)[["win_start", "win_end"]]
    usecols = [id_col, time_col, item_col, value_col]
    keep = []
    reader = pd.read_csv(inner_path, usecols=usecols, dtype=dtype,
                         chunksize=chunksize, low_memory=False)
    n_chunks = 0
    for chunk in tqdm(reader, desc=f"scan {label}", unit="chunk"):
        n_chunks += 1
        chunk = chunk[chunk[item_col].isin(itemset)]
        if chunk.empty:
            continue
        chunk = chunk.dropna(subset=[id_col])
        if str(chunk[id_col].dtype) != "int64":
            chunk[id_col] = pd.to_numeric(chunk[id_col], errors="coerce")
            chunk = chunk.dropna(subset=[id_col])
            chunk[id_col] = chunk[id_col].astype("int64")
        chunk = chunk[chunk[id_col].isin(ids)]
        if chunk.empty:
            continue
        chunk[value_col] = pd.to_numeric(chunk[value_col], errors="coerce")
        chunk = chunk.dropna(subset=[value_col])
        if chunk.empty:
            continue
        chunk[time_col] = pd.to_datetime(chunk[time_col], errors="coerce")
        chunk = chunk.merge(wmap, left_on=id_col, right_index=True, how="left")
        m = (chunk[time_col] >= chunk["win_start"]) & (chunk[time_col] <= chunk["win_end"])
        chunk = chunk.loc[m, [id_col, item_col, value_col, time_col]]
        if value_transforms:
            for iid, fn in value_transforms.items():
                sel = chunk[item_col] == iid
                if sel.any():
                    chunk.loc[sel, value_col] = fn(chunk.loc[sel, value_col])
        if len(chunk):
            keep.append(chunk)
    logger.info("%s: scanned %d chunks, kept %d rows", label, n_chunks,
                sum(len(k) for k in keep))
    if not keep:
        return pd.DataFrame(columns=[id_col, item_col, value_col, time_col])
    return pd.concat(keep, ignore_index=True)

def aggregate_long(events, id_col, item_col, value_col, time_col, itemid_to_feature):
    """Collapse long events to a wide per-id table with min/max/mean/last per feature."""
    if events.empty:
        return pd.DataFrame(columns=[id_col])
    ev = events.copy()
    ev["feature"] = ev[item_col].map(itemid_to_feature)
    ev = ev.dropna(subset=["feature"])
    ev = ev.sort_values(time_col)
    grp = ev.groupby([id_col, "feature"])[value_col]
    agg = grp.agg(["min", "max", "mean"]).reset_index()
    last  = ev.groupby([id_col, "feature"])[value_col].last().reset_index().rename(columns={value_col: "last"})
    first = ev.groupby([id_col, "feature"])[value_col].first().reset_index().rename(columns={value_col: "first"})
    agg = agg.merge(last, on=[id_col, "feature"], how="left").merge(first, on=[id_col, "feature"], how="left")
    agg["slope"] = agg["last"] - agg["first"]      # trajectory delta over the window
    agg = agg.drop(columns=["first"])
    wide = agg.pivot(index=id_col, columns="feature")
    wide.columns = [f"{feat}_{stat}" for stat, feat in wide.columns]
    return wide.reset_index()

logger.info("Utilities loaded.")


21:11:56 | INFO    | Utilities loaded.


---
## Part A. MIMIC-IV v2.2

MIMIC-IV serves as the development and internal temporal-validation database. We build the landmark cohort, attach demographics, comorbidity burden (Elixhauser, Quan 2005 adaptation), first-24h physiology, a first-day SOFA approximation, post-anchor treatments, and outcomes. Every large table is streamed once and filtered to the cohort and to a concept whitelist.


In [5]:
# --- MIMIC-IV concept dictionaries -----------------------------------------
# chartevents itemids -> feature name (weight/height/fio2/gcs folded in so
# chartevents is scanned exactly once).
MIMIC_VITALS = {
    220045: "heart_rate",
    220179: "sbp", 220050: "sbp",
    220180: "dbp", 220051: "dbp",
    220181: "mbp", 220052: "mbp", 225312: "mbp",
    220210: "resp_rate", 224690: "resp_rate",
    223762: "temp", 223761: "temp",          # 223761 is Fahrenheit -> transformed
    220277: "spo2",
    223835: "fio2",                           # per cent -> transformed to fraction
    220739: "gcs_eye", 223900: "gcs_verbal", 223901: "gcs_motor",
    226512: "weight_kg", 224639: "weight_kg",
    226730: "height_cm",
}
MIMIC_VITAL_TRANSFORMS = {
    223761: (lambda s: (s - 32.0) * 5.0 / 9.0),         # F -> C
    223835: (lambda s: s.where(s <= 1.0, s / 100.0)),   # FiO2 % -> fraction
}

# labevents itemids -> feature name (several concepts have chem + blood-gas ids)
MIMIC_LABS = {
    50912: "creatinine", 51006: "bun",
    50983: "sodium", 50824: "sodium",
    50971: "potassium", 50822: "potassium",
    50882: "bicarbonate",
    50902: "chloride", 50806: "chloride",
    50931: "glucose", 50809: "glucose",
    51222: "hemoglobin", 50811: "hemoglobin",
    51221: "hematocrit", 50810: "hematocrit",
    51301: "wbc", 51300: "wbc",
    51265: "platelets",
    50813: "lactate",
    50820: "ph", 50831: "ph",
    50821: "pao2", 50818: "paco2",
    50885: "bilirubin_total", 50862: "albumin",
    51237: "inr", 51275: "ptt", 50868: "anion_gap",
}

# inputevents itemids (vasoactive agents)
MIMIC_PRESSORS = {221906: "norepinephrine", 221289: "epinephrine",
                  221749: "phenylephrine", 221662: "dopamine", 222315: "vasopressin"}
MIMIC_INOTROPES = {221653: "dobutamine", 221986: "milrinone"}

# procedureevents itemids
MIMIC_VENT = {225792: "invasive", 225794: "noninvasive"}
MIMIC_RRT  = {225441: "hemodialysis", 225802: "crrt", 225803: "cvvhd",
              225805: "pd", 225809: "cvvhdf", 225955: "scuf"}

logger.info("MIMIC concept dictionaries: %d vital ids, %d lab ids, %d vaso ids",
            len(MIMIC_VITALS), len(MIMIC_LABS), len(MIMIC_PRESSORS) + len(MIMIC_INOTROPES))


21:11:56 | INFO    | MIMIC concept dictionaries: 20 vital ids, 28 lab ids, 7 vaso ids


In [6]:
# --- Load base tables and build the pre-exclusion stay frame ----------------
with timed("MIMIC base tables"):
    icu = pd.read_csv(mimic_path("icu", "icustays"),
                      usecols=["subject_id", "hadm_id", "stay_id", "intime", "outtime", "los"],
                      parse_dates=["intime", "outtime"])
    pat = pd.read_csv(mimic_path("hosp", "patients"),
                      usecols=["subject_id", "gender", "anchor_age", "anchor_year", "dod"],
                      parse_dates=["dod"])
    adm = pd.read_csv(mimic_path("hosp", "admissions"),
                      usecols=["subject_id", "hadm_id", "admittime", "dischtime", "deathtime",
                               "admission_type", "admission_location", "insurance", "race",
                               "hospital_expire_flag"],
                      parse_dates=["admittime", "dischtime", "deathtime"])

base = (icu.merge(pat, on="subject_id", how="left")
            .merge(adm, on=["subject_id", "hadm_id"], how="left"))

# age at admission (MIMIC anchor scheme)
base["admission_age"] = base["anchor_age"] + (base["admittime"].dt.year - base["anchor_year"])
# precise ICU length of stay in hours
base["los_h"] = (base["outtime"] - base["intime"]).dt.total_seconds() / 3600.0
# decision anchor and windows
base["anchor"]    = base["intime"] + pd.Timedelta(hours=CFG.PRED_HORIZON_H)
base["win_start"] = base["intime"]
base["win_end"]   = base["intime"] + pd.Timedelta(hours=CFG.FEATURE_WINDOW_H)

logger.info("MIMIC raw ICU stays: %d (patients: %d)", len(base), base.subject_id.nunique())
base.head(3)


21:11:56 | INFO    | START  MIMIC base tables
21:11:59 | INFO    | DONE   MIMIC base tables  (3.0s)
21:12:00 | INFO    | MIMIC raw ICU stays: 73181 (patients: 50920)


In [7]:
# --- CONSORT-style exclusion cascade ---------------------------------------
consort = []
def _step(df, label):
    consort.append({"step": label, "stays": int(len(df)), "patients": int(df.subject_id.nunique())})
    return df

df = base.copy()
_step(df, "All ICU stays")

df = df[df["admission_age"] >= CFG.MIN_AGE].copy()
_step(df, f"Adults (age >= {CFG.MIN_AGE})")

sort_keys = ["subject_id", "intime"] if CFG.FIRST_STAY_KEY == "subject" else ["hadm_id", "intime"]
grp_key   = "subject_id" if CFG.FIRST_STAY_KEY == "subject" else "hadm_id"
df = df.sort_values(sort_keys).groupby(grp_key, as_index=False).first()
_step(df, f"First ICU stay per {CFG.FIRST_STAY_KEY}")

df = df[df["los_h"] >= CFG.min_los_h()].copy()
_step(df, f"Alive in ICU at anchor + buffer (LOS >= {CFG.min_los_h():.0f} h)")

df = df[df["hospital_expire_flag"].notna()].copy()
_step(df, "Non-missing in-hospital outcome")

mimic_cohort = df.reset_index(drop=True)
consort_df = pd.DataFrame(consort)
consort_df["excluded"] = consort_df["stays"].shift(1) - consort_df["stays"]
consort_df.to_csv(CFG.TABLE_DIR / "mimic_consort.csv", index=False)
logger.info("MIMIC final cohort: %d stays", len(mimic_cohort))
consort_df


21:12:00 | INFO    | MIMIC final cohort: 49684 stays


,step,stays,patients,excluded
0,All ICU stays,73181,50920,NaN
1,Adults (age >= 18),73181,50920,0.0
2,First ICU stay per subject,50920,50920,22261.0
3,Alive in ICU at anchor + buffer (LOS >= 10 h),49684,49684,1236.0
4,Non-missing in-hospital outcome,49684,49684,0.0


In [8]:
# --- Demographics and outcomes ---------------------------------------------
def harmonise_race(x):
    if not isinstance(x, str):
        return "Other/Unknown"
    u = x.upper()
    if "WHITE" in u: return "White"
    if "BLACK" in u or "AFRICAN" in u: return "Black"
    if "HISPANIC" in u or "LATINO" in u: return "Hispanic"
    if "ASIAN" in u: return "Asian"
    return "Other/Unknown"

mc = mimic_cohort
mc["sex"] = mc["gender"].map({"M": "Male", "F": "Female"}).fillna("Unknown")
mc["race_group"] = mc["race"].map(harmonise_race)

# outcomes
mc["in_hospital_mortality"] = mc["hospital_expire_flag"].astype(int)
mc["icu_mortality"] = ((mc["deathtime"].notna()) & (mc["deathtime"] <= mc["outtime"])).astype(int)
_dod_days = (mc["dod"] - mc["intime"]).dt.days
mc["mortality_28d"] = ((mc["dod"].notna()) & (_dod_days <= 28)).astype(int)
mc["hosp_los_days"] = (mc["dischtime"] - mc["admittime"]).dt.total_seconds() / 86400.0

logger.info("MIMIC in-hospital mortality: %.1f%% | ICU mortality: %.1f%%",
            100 * mc["in_hospital_mortality"].mean(), 100 * mc["icu_mortality"].mean())
mc[["stay_id", "admission_age", "sex", "race_group", "los_h",
    "in_hospital_mortality", "icu_mortality", "mortality_28d"]].head()


21:12:00 | INFO    | MIMIC in-hospital mortality: 9.8% | ICU mortality: 6.5%


In [9]:
# --- Elixhauser comorbidity mapping (Quan 2005 adaptation) ------------------
# Prefix-based ICD-9-CM / ICD-10 mapping bundled for reproducibility. Codes are
# normalised (dots stripped, upper-cased) and matched by prefix. van Walraven
# weights provide a single continuous burden score.
ELIX = {
 "chf":            {"9": ["39891","40201","40211","40291","40401","40403","40411","40413","40491","40493","4254","4255","4257","4258","4259","428"], "10": ["I099","I110","I130","I132","I255","I420","I425","I426","I427","I428","I429","I43","I50","P290"]},
 "arrhythmia":     {"9": ["4260","42613","4267","4269","42610","42612","4270","4271","4272","4273","4274","4276","4278","4279","7850","99601","99604","V450","V533"], "10": ["I441","I442","I443","I456","I459","I47","I48","I49","R000","R001","R008","T821","Z450","Z950"]},
 "valvular":       {"9": ["0932","394","395","396","397","424","7463","7464","7465","7466","V422","V433"], "10": ["A520","I05","I06","I07","I08","I091","I098","I34","I35","I36","I37","I38","I39","Q230","Q231","Q232","Q233","Z952","Z953","Z954"]},
 "pulm_circ":      {"9": ["4150","4151","416","4170","4178","4179"], "10": ["I26","I27","I280","I288","I289"]},
 "pvd":            {"9": ["0930","4373","440","441","4431","4432","4438","4439","4471","5571","5579","V434"], "10": ["I70","I71","I731","I738","I739","I771","I790","I792","K551","K558","K559","Z958","Z959"]},
 "htn_uncomp":     {"9": ["401"], "10": ["I10"]},
 "htn_comp":       {"9": ["402","403","404","405"], "10": ["I11","I12","I13","I15"]},
 "paralysis":      {"9": ["3341","342","343","3440","3441","3442","3443","3444","3445","3446","3449"], "10": ["G041","G114","G801","G802","G81","G82","G830","G831","G832","G833","G834","G839"]},
 "neuro_other":    {"9": ["3319","3320","3321","3334","3335","33392","334","335","3362","340","341","345","3481","3483","78031","78032","78033","78039","7803","7843"], "10": ["G10","G11","G12","G13","G20","G21","G22","G254","G255","G312","G318","G319","G32","G35","G36","G37","G40","G41","G931","G934","R470","R56"]},
 "chronic_pulm":   {"9": ["4168","4169","490","491","492","493","494","495","496","500","501","502","503","504","505","5064","5081","5088"], "10": ["I278","I279","J40","J41","J42","J43","J44","J45","J46","J47","J60","J61","J62","J63","J64","J65","J66","J67","J684","J701","J703"]},
 "diab_uncomp":    {"9": ["2500","2501","2502","2503"], "10": ["E100","E101","E109","E110","E111","E119","E120","E121","E129","E130","E131","E139","E140","E141","E149"]},
 "diab_comp":      {"9": ["2504","2505","2506","2507","2508","2509"], "10": ["E102","E103","E104","E105","E106","E107","E108","E112","E113","E114","E115","E116","E117","E118","E132","E133","E134","E135","E136","E137","E138","E142","E143","E144","E145","E146","E147","E148"]},
 "hypothyroid":    {"9": ["2409","243","244","2461","2468"], "10": ["E00","E01","E02","E03","E890"]},
 "renal_failure":  {"9": ["40301","40311","40391","40402","40403","40412","40413","40492","40493","585","586","5880","V420","V451","V56"], "10": ["I120","I131","N18","N19","N250","Z490","Z491","Z492","Z940","Z992"]},
 "liver":          {"9": ["07022","07023","07032","07033","07044","07054","0706","0709","456","5710","5712","5713","5714","5715","5716","5718","5719","5723","5728","5735","5738","5739","V427"], "10": ["B18","I85","I864","I982","K70","K711","K713","K714","K715","K717","K72","K73","K74","K760","K762","K763","K764","K765","K766","K767","K768","K769","Z944"]},
 "pud":            {"9": ["5317","5319","5327","5329","5337","5339","5347","5349"], "10": ["K257","K259","K267","K269","K277","K279","K287","K289"]},
 "hiv":            {"9": ["042","043","044"], "10": ["B20","B21","B22","B24"]},
 "lymphoma":       {"9": ["200","201","202","2030","2386"], "10": ["C81","C82","C83","C84","C85","C88","C96","C900","C902"]},
 "mets":           {"9": ["196","197","198","199"], "10": ["C77","C78","C79","C80"]},
 "solid_tumour":   {"9": ["140","141","142","143","144","145","146","147","148","149","150","151","152","153","154","155","156","157","158","159","160","161","162","163","164","165","166","167","168","169","170","171","172","174","175","176","177","178","179","180","181","182","183","184","185","186","187","188","189","190","191","192","193","194","195"], "10": ["C0","C1","C2","C30","C31","C32","C33","C34","C37","C38","C39","C40","C41","C43","C45","C46","C47","C48","C49","C50","C51","C52","C53","C54","C55","C56","C57","C58","C60","C61","C62","C63","C64","C65","C66","C67","C68","C69","C70","C71","C72","C73","C74","C75","C76","C97"]},
 "rheum":          {"9": ["7010","7100","7101","7102","7103","7104","7108","7109","7112","714","7193","720","725","7285","72889","72930"], "10": ["L940","L941","L943","M05","M06","M08","M120","M123","M30","M31","M32","M33","M34","M35","M45","M461","M468","M469"]},
 "coagulopathy":   {"9": ["286","2871","2873","2874","2875"], "10": ["D65","D66","D67","D68","D691","D693","D694","D695","D696"]},
 "obesity":        {"9": ["2780"], "10": ["E66"]},
 "weight_loss":    {"9": ["260","261","262","263","7832","7994"], "10": ["E40","E41","E42","E43","E44","E45","E46","R634","R64"]},
 "fluid_electro":  {"9": ["2536","276"], "10": ["E222","E86","E87"]},
 "blood_loss_anem":{"9": ["2800"], "10": ["D500"]},
 "def_anemia":     {"9": ["2801","2808","2809","281"], "10": ["D508","D509","D51","D52","D53"]},
 "alcohol":        {"9": ["2652","2911","2912","2913","2915","2918","2919","3030","3039","3050","3575","4255","5353","5710","5711","5712","5713","980","V113"], "10": ["F10","E52","G621","I426","K292","K700","K703","K709","T51","Z502","Z714","Z721"]},
 "drug_abuse":     {"9": ["292","304","3052","3053","3054","3055","3056","3057","3058","3059","V6542"], "10": ["F11","F12","F13","F14","F15","F16","F18","F19","Z715","Z722"]},
 "psychoses":      {"9": ["2938","295","29604","29614","29644","29654","297","298"], "10": ["F20","F22","F23","F24","F25","F28","F29","F302","F312","F315"]},
 "depression":     {"9": ["2962","2963","2965","3004","309","311"], "10": ["F204","F313","F314","F315","F32","F33","F341","F412","F432"]},
}
# van Walraven weights (integer points per Elixhauser category)
VW_WEIGHTS = {"chf":7,"arrhythmia":5,"valvular":-1,"pulm_circ":4,"pvd":2,"htn_uncomp":0,"htn_comp":0,
 "paralysis":7,"neuro_other":6,"chronic_pulm":3,"diab_uncomp":0,"diab_comp":0,"hypothyroid":0,
 "renal_failure":5,"liver":11,"pud":0,"hiv":0,"lymphoma":9,"mets":12,"solid_tumour":4,"rheum":0,
 "coagulopathy":3,"obesity":-4,"weight_loss":6,"fluid_electro":5,"blood_loss_anem":-2,"def_anemia":-2,
 "alcohol":0,"drug_abuse":-7,"psychoses":0,"depression":-3}

def _norm_code(code):
    return str(code).replace(".", "").replace(" ", "").upper()

# Pre-compile prefix lookup per version for speed
_ELIX_PREFIX = {"9": {}, "10": {}}
for cat, vmap in ELIX.items():
    for ver in ("9", "10"):
        for pref in vmap.get(ver, []):
            _ELIX_PREFIX[ver].setdefault(pref.upper(), []).append(cat)

def elix_categories_for(code, version):
    c = _norm_code(code)
    ver = "10" if str(version) == "10" else "9"
    hits = set()
    table = _ELIX_PREFIX[ver]
    # match by decreasing prefix length is unnecessary; collect all prefix hits
    for pref, cats in table.items():
        if c.startswith(pref):
            hits.update(cats)
    return hits

logger.info("Elixhauser mapping compiled: %d categories", len(ELIX))


21:12:00 | INFO    | Elixhauser mapping compiled: 31 categories


In [10]:
# --- Comorbidity flags from diagnoses_icd ----------------------------------
if cache_exists("mimic_comorbid"):
    mimic_comorbid = load_cache("mimic_comorbid")
    logger.info("loaded mimic_comorbid from cache")
else:
    with timed("MIMIC comorbidities"):
        dx = pd.read_csv(mimic_path("hosp", "diagnoses_icd"),
                         usecols=["hadm_id", "icd_code", "icd_version"])
        dx = dx[dx["hadm_id"].isin(set(mc["hadm_id"]))].copy()
        # map each diagnosis code to Elixhauser categories
        cat_rows = []
        # vectorise by unique (code, version) to avoid recomputing
        uniq = dx[["icd_code", "icd_version"]].drop_duplicates()
        code2cats = {}
        for _, r in uniq.iterrows():
            code2cats[(r.icd_code, r.icd_version)] = elix_categories_for(r.icd_code, r.icd_version)
        dx["cats"] = dx.apply(lambda r: code2cats.get((r.icd_code, r.icd_version), set()), axis=1)
        # explode to hadm x category flags
        exploded = dx.explode("cats").dropna(subset=["cats"])
        flags = (exploded.assign(v=1)
                 .pivot_table(index="hadm_id", columns="cats", values="v",
                              aggfunc="max", fill_value=0))
        for cat in ELIX:
            if cat not in flags.columns:
                flags[cat] = 0
        flags = flags[list(ELIX.keys())].astype("int8")
        flags["elix_count"] = flags.sum(axis=1).astype("int16")
        flags["van_walraven"] = sum(flags[cat] * w for cat, w in VW_WEIGHTS.items()).astype("int16")
        mimic_comorbid = flags.reset_index()
        save_cache(mimic_comorbid, "mimic_comorbid")

logger.info("MIMIC comorbidity table: %d admissions, median Elixhauser count = %.0f",
            len(mimic_comorbid), mimic_comorbid["elix_count"].median() if len(mimic_comorbid) else 0)
mimic_comorbid.head(3)


21:12:00 | INFO    | loaded mimic_comorbid from cache
21:12:00 | INFO    | MIMIC comorbidity table: 47609 admissions, median Elixhauser count = 4


### First-24h physiology and post-anchor treatments

The two large event tables (chartevents and labevents) are each streamed once. chartevents is filtered to the vital, GCS component, FiO2, weight, and height concepts; labevents to the chemistry, haematology, and blood-gas concepts. Both are restricted to the pre-anchor window. Treatments are read from inputevents (vasoactive agents) and procedureevents (ventilation, renal replacement) and are attributed to the post-anchor window when they extend beyond `t0 + 24h`.


In [11]:
# --- Vitals / GCS / FiO2 / weight / height from chartevents (single pass) ---
if cache_exists("mimic_vitals"):
    mimic_vitals = load_cache("mimic_vitals")
    logger.info("loaded mimic_vitals from cache")
else:
    with timed("MIMIC chartevents extraction"):
        win = mc[["stay_id", "win_start", "win_end"]].copy()
        ev = extract_events_windowed(
            mimic_path("icu", "chartevents"),
            id_col="stay_id", time_col="charttime", item_col="itemid", value_col="valuenum",
            itemids=list(MIMIC_VITALS.keys()), windows=win, chunksize=CFG.CHUNK_LARGE,
            dtype={"stay_id": "int64", "itemid": "int64"},
            value_transforms=MIMIC_VITAL_TRANSFORMS, label="chartevents")
        mimic_vitals = aggregate_long(ev, "stay_id", "itemid", "valuenum", "charttime", MIMIC_VITALS)
        save_cache(mimic_vitals, "mimic_vitals")
        del ev; gc.collect()

logger.info("MIMIC vitals wide table: %d stays x %d columns", *mimic_vitals.shape)
mimic_vitals.head(3)


21:12:00 | INFO    | loaded mimic_vitals from cache
21:12:00 | INFO    | MIMIC vitals wide table: 48955 stays x 66 columns


In [12]:
# --- Labs from labevents (single pass; keyed by hadm_id) --------------------
if cache_exists("mimic_labs"):
    mimic_labs = load_cache("mimic_labs")
    logger.info("loaded mimic_labs from cache")
else:
    with timed("MIMIC labevents extraction"):
        # labevents carries hadm_id; align its window to the ICU intime of the cohort
        winl = mc[["hadm_id", "win_start", "win_end"]].copy()
        ev = extract_events_windowed(
            mimic_path("hosp", "labevents"),
            id_col="hadm_id", time_col="charttime", item_col="itemid", value_col="valuenum",
            itemids=list(MIMIC_LABS.keys()), windows=winl, chunksize=CFG.CHUNK_LARGE,
            dtype={"itemid": "int64"}, label="labevents")
        labs_h = aggregate_long(ev, "hadm_id", "itemid", "valuenum", "charttime", MIMIC_LABS)
        # bring back to stay grain via the cohort hadm_id -> stay_id mapping
        mimic_labs = mc[["stay_id", "hadm_id"]].merge(labs_h, on="hadm_id", how="left").drop(columns=["hadm_id"])
        save_cache(mimic_labs, "mimic_labs")
        del ev, labs_h; gc.collect()

logger.info("MIMIC labs wide table: %d stays x %d columns", *mimic_labs.shape)
mimic_labs.head(3)


21:12:01 | INFO    | loaded mimic_labs from cache
21:12:01 | INFO    | MIMIC labs wide table: 49684 stays x 101 columns


In [13]:
# --- Post-anchor treatments: vasopressors, ventilation, renal replacement ---
def _load_filtered(path, usecols, dtype, itemids, id_col, chunksize, label):
    ids = set(mc[id_col]); its = set(itemids); parts = []
    for chunk in tqdm(pd.read_csv(path, usecols=usecols, dtype=dtype,
                                  chunksize=chunksize, low_memory=False),
                      desc=f"scan {label}", unit="chunk"):
        chunk = chunk[chunk["itemid"].isin(its)]
        if chunk.empty:
            continue
        chunk = chunk[chunk[id_col].isin(ids)]
        if len(chunk):
            parts.append(chunk)
    return pd.concat(parts, ignore_index=True) if parts else pd.DataFrame(columns=usecols)

if cache_exists("mimic_treat"):
    mimic_treat = load_cache("mimic_treat")
    logger.info("loaded mimic_treat from cache")
else:
    with timed("MIMIC treatment extraction"):
        anchor = mc.set_index("stay_id")["anchor"]
        w_end  = mc.set_index("stay_id")["win_end"]

        # vasoactive agents (inputevents)
        ie = _load_filtered(mimic_path("icu", "inputevents"),
                            ["stay_id", "starttime", "endtime", "itemid", "rate"],
                            {"stay_id": "int64", "itemid": "int64"},
                            list(MIMIC_PRESSORS) + list(MIMIC_INOTROPES),
                            "stay_id", CFG.CHUNK_MED, "inputevents")
        ie["starttime"] = pd.to_datetime(ie["starttime"], errors="coerce")
        ie["endtime"]   = pd.to_datetime(ie["endtime"], errors="coerce")
        ie["anchor"]    = ie["stay_id"].map(anchor)
        ie["win_end"]   = ie["stay_id"].map(w_end)
        ie["is_pressor"]  = ie["itemid"].isin(MIMIC_PRESSORS)
        ie["is_inotrope"] = ie["itemid"].isin(MIMIC_INOTROPES)
        ie["post"]  = ie["endtime"] > ie["anchor"]
        ie["day1"]  = ie["starttime"] < ie["win_end"]

        # ventilation and renal replacement (procedureevents)
        pe = _load_filtered(mimic_path("icu", "procedureevents"),
                            ["stay_id", "starttime", "endtime", "itemid"],
                            {"stay_id": "int64", "itemid": "int64"},
                            list(MIMIC_VENT) + list(MIMIC_RRT),
                            "stay_id", CFG.CHUNK_MED, "procedureevents")
        pe["starttime"] = pd.to_datetime(pe["starttime"], errors="coerce")
        pe["endtime"]   = pd.to_datetime(pe["endtime"], errors="coerce")
        pe["anchor"]    = pe["stay_id"].map(anchor)
        pe["win_end"]   = pe["stay_id"].map(w_end)
        pe["is_vent"] = pe["itemid"].isin(MIMIC_VENT)
        pe["is_rrt"]  = pe["itemid"].isin(MIMIC_RRT)
        pe["is_invasive_vent"] = pe["itemid"] == 225792
        pe["post"] = pe["endtime"] > pe["anchor"]
        pe["day1"] = pe["starttime"] < pe["win_end"]

        def _grp_any(frame, mask, id_col="stay_id"):
            s = frame.loc[mask].groupby(id_col).size()
            return s.index[s > 0]

        idx = mc[["stay_id"]].copy()
        def _flag(ids):
            return idx["stay_id"].isin(set(ids)).astype("int8")

        idx["vaso_post"]     = _flag(_grp_any(ie, ie.is_pressor & ie.post))
        idx["vaso_day1"]     = _flag(_grp_any(ie, ie.is_pressor & ie.day1))
        idx["inotrope_post"] = _flag(_grp_any(ie, ie.is_inotrope & ie.post))
        idx["vent_post"]     = _flag(_grp_any(pe, pe.is_vent & pe.post))
        idx["vent_day1"]     = _flag(_grp_any(pe, pe.is_vent & pe.day1))
        idx["invasive_vent_post"] = _flag(_grp_any(pe, pe.is_invasive_vent & pe.post))
        idx["rrt_post"]      = _flag(_grp_any(pe, pe.is_rrt & pe.post))
        idx["rrt_day1"]      = _flag(_grp_any(pe, pe.is_rrt & pe.day1))

        # number of distinct pressor agents used post-anchor (treatment intensity)
        npress = (ie.loc[ie.is_pressor & ie.post]
                    .groupby("stay_id")["itemid"].nunique().rename("n_pressor_types_post"))
        idx = idx.merge(npress, on="stay_id", how="left")
        idx["n_pressor_types_post"] = idx["n_pressor_types_post"].fillna(0).astype("int8")

        # composite: any post-anchor organ support (the actions a score would move)
        idx["any_organ_support_post"] = ((idx.vaso_post | idx.vent_post | idx.rrt_post)).astype("int8")

        mimic_treat = idx
        save_cache(mimic_treat, "mimic_treat")
        del ie, pe; gc.collect()

logger.info("MIMIC post-anchor treatment rates: vaso %.1f%% | vent %.1f%% | rrt %.1f%%",
            100 * mimic_treat.vaso_post.mean(), 100 * mimic_treat.vent_post.mean(),
            100 * mimic_treat.rrt_post.mean())
mimic_treat.head(3)


21:12:01 | INFO    | loaded mimic_treat from cache
21:12:01 | INFO    | MIMIC post-anchor treatment rates: vaso 27.6% | vent 38.5% | rrt 3.7%


In [14]:
# --- First-day SOFA approximation ------------------------------------------
# Documented approximation: organ subscores from worst first-24h physiology and
# day-1 vasoactive/ventilation support. Missing components contribute 0 and are
# tracked via sofa_coverage so downstream analyses can condition on completeness.
def _score(series, bins, labels):
    return pd.cut(series, bins=bins, labels=labels, right=True).astype("float").fillna(0)

def compute_sofa(vit, lab, treat):
    v = vit.set_index("stay_id"); l = lab.set_index("stay_id"); t = treat.set_index("stay_id")
    idx = treat["stay_id"]
    def col(df, name):
        return df[name].reindex(idx).values if name in df.columns else np.full(len(idx), np.nan)

    pao2 = col(l, "pao2_min"); fio2 = col(v, "fio2_max")
    with np.errstate(divide="ignore", invalid="ignore"):
        pf = np.where((fio2 is not None) & (np.asarray(fio2) > 0),
                      np.asarray(pao2) / np.asarray(fio2), np.nan)
    vent1 = col(t, "vent_day1")
    resp = np.select(
        [pf < 100, pf < 200, pf < 300, pf < 400],
        [np.where(np.nan_to_num(vent1) == 1, 4, 2),
         np.where(np.nan_to_num(vent1) == 1, 3, 2), 2, 1], default=0)
    resp = np.where(np.isnan(pf), 0, resp)

    plt_min = col(l, "platelets_min")
    coag = np.select([plt_min < 20, plt_min < 50, plt_min < 100, plt_min < 150],
                     [4, 3, 2, 1], default=0); coag = np.where(np.isnan(plt_min), 0, coag)

    bili = col(l, "bilirubin_total_max")
    liver = np.select([bili >= 12, bili >= 6, bili >= 2, bili >= 1.2],
                      [4, 3, 2, 1], default=0); liver = np.where(np.isnan(bili), 0, liver)

    mbp = col(v, "mbp_min"); vaso1 = np.nan_to_num(col(t, "vaso_day1")); ino1 = np.nan_to_num(col(t, "inotrope_post"))
    cardio = np.where(vaso1 == 1, 3, np.where((mbp < 70) | (ino1 == 1), 1, 0))
    cardio = np.where(np.isnan(mbp) & (vaso1 == 0), 0, cardio)

    gcs = (col(v, "gcs_eye_min") + col(v, "gcs_verbal_min") + col(v, "gcs_motor_min"))
    cns = np.select([gcs < 6, gcs < 10, gcs < 13, gcs < 15], [4, 3, 2, 1], default=0)
    cns = np.where(np.isnan(gcs), 0, cns)

    cr = col(l, "creatinine_max")
    renal = np.select([cr >= 5, cr >= 3.5, cr >= 2, cr >= 1.2], [4, 3, 2, 1], default=0)
    renal = np.where(np.isnan(cr), 0, renal)

    comps = np.vstack([resp, coag, liver, cardio, cns, renal]).astype(float)
    coverage = np.vstack([~np.isnan(pf), ~np.isnan(plt_min), ~np.isnan(bili),
                          ~np.isnan(mbp) | (vaso1 == 1), ~np.isnan(gcs), ~np.isnan(cr)]).mean(axis=0)
    out = pd.DataFrame({"stay_id": idx.values,
                        "sofa_resp": resp, "sofa_coag": coag, "sofa_liver": liver,
                        "sofa_cardio": cardio, "sofa_cns": cns, "sofa_renal": renal,
                        "sofa_total": comps.sum(axis=0), "sofa_coverage": coverage})
    return out

mimic_sofa = compute_sofa(mimic_vitals, mimic_labs, mimic_treat)
logger.info("MIMIC first-day SOFA: median %.0f (IQR %.0f-%.0f), mean coverage %.2f",
            mimic_sofa.sofa_total.median(),
            mimic_sofa.sofa_total.quantile(.25), mimic_sofa.sofa_total.quantile(.75),
            mimic_sofa.sofa_coverage.mean())
mimic_sofa.head(3)


21:12:01 | INFO    | MIMIC first-day SOFA: median 3 (IQR 1-7), mean coverage 0.66


In [15]:
# --- Assemble the MIMIC analytic table and fix the canonical schema ---------
# The canonical column dictionary is derived deterministically from the concept
# dictionaries, so both databases reindex to an identical set of columns and any
# concept absent in one database becomes an explicit missing column rather than a
# silent schema divergence.
STATS = ["min", "max", "mean", "last", "slope"]
VITAL_FEATURES = sorted(set(MIMIC_VITALS.values()))
LAB_FEATURES   = sorted(set(MIMIC_LABS.values()))
PHYSIO_COLUMNS = [f"{f}_{s}" for f in (VITAL_FEATURES + LAB_FEATURES) for s in STATS]
SOFA_COLUMNS   = ["sofa_resp", "sofa_coag", "sofa_liver", "sofa_cardio",
                  "sofa_cns", "sofa_renal", "sofa_total", "sofa_coverage"]
TREAT_COLUMNS  = ["vaso_post", "vaso_day1", "inotrope_post", "vent_post", "vent_day1",
                  "invasive_vent_post", "rrt_post", "rrt_day1",
                  "n_pressor_types_post", "any_organ_support_post"]
COMORBID_COLUMNS = list(ELIX.keys()) + ["elix_count", "van_walraven"]
DEMO_COLUMNS   = ["admission_age", "sex", "race_group", "los_h", "admission_type", "insurance"]
OUTCOME_COLUMNS = ["in_hospital_mortality", "icu_mortality", "mortality_28d", "hosp_los_days"]
ID_COLUMNS     = ["db", "patient_id", "hadm_id"]
CANON = (ID_COLUMNS + DEMO_COLUMNS + OUTCOME_COLUMNS + PHYSIO_COLUMNS
         + SOFA_COLUMNS + TREAT_COLUMNS + COMORBID_COLUMNS)

def finalise_schema(df, db_label):
    """Return df with exactly the canonical columns in canonical order.
    Missing concepts are added as NaN; treatment and comorbidity flags are
    coerced to 0 (absence), which is their correct interpretation."""
    df = df.copy()
    df["db"] = db_label
    for c in CANON:
        if c not in df.columns:
            df[c] = np.nan
    for c in TREAT_COLUMNS + COMORBID_COLUMNS:
        df[c] = pd.to_numeric(df[c], errors="coerce").fillna(0)
    df = df[CANON]
    return df

# demographics + outcomes base (rename stay grain to the harmonised key)
_demo = ["stay_id", "hadm_id", "admission_age", "sex", "race_group", "los_h",
         "admission_type", "insurance", "in_hospital_mortality", "icu_mortality",
         "mortality_28d", "hosp_los_days"]
mimic_asm = mc[_demo].rename(columns={"stay_id": "patient_id"})
# comorbidity burden joins on hadm_id
mimic_asm = mimic_asm.merge(mimic_comorbid, on="hadm_id", how="left")
# physiology, labs, treatments, SOFA join on the stay grain
for _tbl in (mimic_vitals, mimic_labs, mimic_treat, mimic_sofa):
    mimic_asm = mimic_asm.merge(_tbl.rename(columns={"stay_id": "patient_id"}),
                                on="patient_id", how="left")

mimic_analytic = finalise_schema(mimic_asm, "MIMIC")
mimic_analytic = downcast(mimic_analytic)
save_cache(mimic_analytic, "mimic_analytic")
mimic_analytic.to_parquet(CFG.OUTPUT_DIR / "mimic_analytic.parquet", index=False)
logger.info("MIMIC analytic table: %d rows x %d cols (%d physiology, %d comorbidity)",
            len(mimic_analytic), mimic_analytic.shape[1], len(PHYSIO_COLUMNS), len(COMORBID_COLUMNS))
mimic_analytic.head(3)


21:12:04 | INFO    | cached mimic_analytic               rows=49684    cols=229
21:12:05 | INFO    | MIMIC analytic table: 49684 rows x 229 cols (165 physiology, 33 comorbidity)


---
## Part B. eICU Collaborative Research Database v2.0

eICU is the external-validation database. It is harmonised into the identical canonical schema fixed in Part A. Three structural differences from MIMIC are handled explicitly. First, eICU records time as an integer offset in minutes from unit admission, so the pre-anchor feature window is the offset interval `[0, PRED_HORIZON_H x 60]` and the post-anchor treatment window is offsets beyond that. Second, high-frequency vitals are stored as wide numeric columns in vitalPeriodic and vitalAperiodic rather than as an item dictionary, so they are aggregated column-wise. Third, Glasgow Coma Scale components and the inspired-oxygen fraction are taken from the APACHE physiology extract (apacheApsVar), where they are recorded reliably over the same first-day window. Treatments are parsed from the treatment and infusionDrug tables by concept string.


In [16]:
# --- eICU concept dictionaries and offset windows ---------------------------
EICU_ANCHOR_MIN = CFG.PRED_HORIZON_H * 60          # decision anchor in minutes
EICU_WIN_LO, EICU_WIN_HI = 0, EICU_ANCHOR_MIN      # pre-anchor feature window

# wide vitals: source column -> harmonised feature
EICU_PERIODIC = {"heartrate": "heart_rate", "respiration": "resp_rate", "sao2": "spo2",
                 "temperature": "temp", "systemicsystolic": "sbp",
                 "systemicdiastolic": "dbp", "systemicmean": "mbp"}
EICU_APERIODIC = {"noninvasivesystolic": "sbp", "noninvasivediastolic": "dbp",
                  "noninvasivemean": "mbp"}

# lab names (as spelled in eICU) -> harmonised feature; variants folded in
EICU_LABS = {
    "creatinine": "creatinine", "BUN": "bun",
    "sodium": "sodium", "potassium": "potassium",
    "bicarbonate": "bicarbonate", "HCO3": "bicarbonate",
    "chloride": "chloride", "glucose": "glucose", "bedside glucose": "glucose",
    "Hgb": "hemoglobin", "Hct": "hematocrit",
    "WBC x 1000": "wbc", "platelets x 1000": "platelets",
    "lactate": "lactate", "pH": "ph",
    "paO2": "pao2", "paCO2": "paco2",
    "total bilirubin": "bilirubin_total", "albumin": "albumin",
    "PT - INR": "inr", "PTT": "ptt", "anion gap": "anion_gap",
}

# treatment / infusion concept keywords (lower case, substring match)
EICU_VASO_AGENTS = {"norepinephrine": ["norepinephrine", "levophed", "norepi"],
                    "epinephrine":    ["epinephrine", "adrenaline"],
                    "phenylephrine":  ["phenylephrine", "neo-synephrine", "neosynephrine"],
                    "dopamine":       ["dopamine"],
                    "vasopressin":    ["vasopressin", "pitressin"]}
EICU_VASO_GENERIC = ["vasopressor", "vasopressors"]
EICU_INOTROPE_KW  = ["dobutamine", "dobutrex", "milrinone", "primacor", "inotrop"]
EICU_VENT_KW      = ["ventilation", "ventilator", "intubation", "intubated", "extubation"]
EICU_INVVENT_KW   = ["mechanical ventilation", "intubation", "intubated"]
EICU_RRT_KW       = ["dialysis", "crrt", "cvvh", "cvvhd", "cavhd", "hemodialysis",
                     "renal replacement", "ultrafiltration", "hemofiltration"]

EICU_RACE = {"Caucasian": "White", "African American": "Black", "Hispanic": "Hispanic",
             "Asian": "Asian", "Native American": "Other/Unknown",
             "Other/Unknown": "Other/Unknown"}

def _eicu_present(path, wanted):
    """Return the subset of wanted columns actually present in a table header."""
    hdr = pd.read_csv(path, nrows=0).columns.tolist()
    return [c for c in wanted if c in hdr]

logger.info("eICU dictionaries ready: %d periodic, %d aperiodic, %d lab names",
            len(EICU_PERIODIC), len(EICU_APERIODIC), len(EICU_LABS))


21:12:05 | INFO    | eICU dictionaries ready: 7 periodic, 3 aperiodic, 22 lab names


In [17]:
# --- eICU aggregation helpers (offset-windowed, memory-bounded) -------------
def _eicu_finalise_wide(parts, id_col):
    """Combine per-chunk partial aggregates into a wide feature_{min,max,mean,last}
    table. `parts` is a list of frames with columns
    [id_col, feature, vmin, vmax, vsum, vcount, last_off, last_val]."""
    if not parts:
        return pd.DataFrame(columns=[id_col])
    allp = pd.concat(parts, ignore_index=True)
    g = allp.groupby([id_col, "feature"], observed=True)
    out = g.agg(vmin=("vmin", "min"), vmax=("vmax", "max"),
                vsum=("vsum", "sum"), vcount=("vcount", "sum")).reset_index()
    lastrows = (allp.sort_values("last_off")
                    .groupby([id_col, "feature"], observed=True)
                    .tail(1)[[id_col, "feature", "last_val"]])
    out = out.merge(lastrows, on=[id_col, "feature"], how="left")
    out["mean"] = out["vsum"] / out["vcount"].replace(0, np.nan)
    out = out.rename(columns={"vmin": "min", "vmax": "max", "last_val": "last"})
    wide = out.pivot(index=id_col, columns="feature", values=["min", "max", "mean", "last"])
    wide.columns = [f"{feat}_{stat}" for stat, feat in wide.columns]
    return wide.reset_index()

def _chunk_partials(df_long, id_col, off_col):
    """Per-chunk aggregate of a long [id, feature, value, offset] frame."""
    g = df_long.groupby([id_col, "feature"], observed=True)["value"]
    cagg = g.agg(vmin="min", vmax="max", vsum="sum", vcount="count").reset_index()
    last = (df_long.sort_values(off_col)
                   .groupby([id_col, "feature"], observed=True)
                   .tail(1)[[id_col, "feature", off_col, "value"]]
                   .rename(columns={off_col: "last_off", "value": "last_val"}))
    return cagg.merge(last, on=[id_col, "feature"], how="left")

def eicu_collect_wide(path, colmap, ids, chunksize, label):
    """Stream a wide eICU vitals table; keep cohort ids and the pre-anchor offset
    window; return a list of per-chunk partial aggregates in long form."""
    id_col, off_col = "patientunitstayid", "observationoffset"
    present = _eicu_present(path, list(colmap.keys()))
    if not present:
        logger.warning("eICU %s: none of the expected value columns present", label)
        return []
    usecols = [id_col, off_col] + present
    idset = set(ids)
    parts = []
    reader = pd.read_csv(path, usecols=lambda c: c in usecols,
                         chunksize=chunksize, low_memory=False)
    for chunk in tqdm(reader, desc=f"scan {label}", unit="chunk"):
        chunk = chunk[chunk[id_col].isin(idset)]
        if chunk.empty:
            continue
        chunk[off_col] = pd.to_numeric(chunk[off_col], errors="coerce")
        chunk = chunk[(chunk[off_col] >= EICU_WIN_LO) & (chunk[off_col] <= EICU_WIN_HI)]
        if chunk.empty:
            continue
        long = chunk.melt(id_vars=[id_col, off_col], value_vars=present,
                          var_name="col", value_name="value")
        long["value"] = pd.to_numeric(long["value"], errors="coerce")
        long = long.dropna(subset=["value"])
        if long.empty:
            continue
        long["feature"] = long["col"].map(colmap)
        long = long.dropna(subset=["feature"])
        parts.append(_chunk_partials(long, id_col, off_col))
    return parts

def eicu_collect_long(path, id_col, off_col, name_col, val_col, name_map, ids, chunksize, label):
    """Stream a long eICU table (lab); map concept name -> feature; keep cohort
    ids and pre-anchor window; return per-chunk partial aggregates."""
    idset = set(ids); namekeys = set(name_map.keys())
    parts = []
    reader = pd.read_csv(path, usecols=[id_col, off_col, name_col, val_col],
                         chunksize=chunksize, low_memory=False)
    for chunk in tqdm(reader, desc=f"scan {label}", unit="chunk"):
        chunk = chunk[chunk[name_col].isin(namekeys)]
        if chunk.empty:
            continue
        chunk = chunk[chunk[id_col].isin(idset)]
        if chunk.empty:
            continue
        chunk[off_col] = pd.to_numeric(chunk[off_col], errors="coerce")
        chunk = chunk[(chunk[off_col] >= EICU_WIN_LO) & (chunk[off_col] <= EICU_WIN_HI)]
        if chunk.empty:
            continue
        chunk["value"] = pd.to_numeric(chunk[val_col], errors="coerce")
        chunk = chunk.dropna(subset=["value"])
        if chunk.empty:
            continue
        chunk["feature"] = chunk[name_col].map(name_map)
        long = chunk[[id_col, off_col, "feature", "value"]]
        parts.append(_chunk_partials(long, id_col, off_col))
    return parts

logger.info("eICU aggregation helpers loaded.")


21:12:06 | INFO    | eICU aggregation helpers loaded.


In [18]:
# --- eICU cohort construction and CONSORT accounting ------------------------
with timed("eICU patient table"):
    _pcols = ["patientunitstayid", "patienthealthsystemstayid", "uniquepid", "gender",
              "age", "ethnicity", "unittype", "unitvisitnumber",
              "hospitaldischargeoffset", "unitdischargeoffset",
              "unitdischargestatus", "hospitaldischargestatus"]
    pt = pd.read_csv(eicu_path("patient"), usecols=lambda c: c in _pcols, low_memory=False)

def _eicu_age(a):
    if isinstance(a, str):
        a = a.strip()
        if a == "":
            return np.nan
        if a.startswith(">"):
            return 90.0
        try:
            return float(a)
        except Exception:
            return np.nan
    return float(a) if pd.notna(a) else np.nan

pt["admission_age"] = pt["age"].map(_eicu_age)
pt["los_h"] = pd.to_numeric(pt["unitdischargeoffset"], errors="coerce") / 60.0

econsort = []
def _estep(df, label):
    econsort.append({"step": label, "stays": int(len(df)),
                     "patients": int(df["uniquepid"].nunique())})
    return df

d = pt.copy();                                            _estep(d, "All ICU stays")
d = d[d["admission_age"] >= CFG.MIN_AGE].copy();           _estep(d, f"Adults (age >= {CFG.MIN_AGE})")
d = (d.sort_values(["uniquepid", "unitvisitnumber", "patientunitstayid"])
       .groupby("uniquepid", as_index=False).first());    _estep(d, "First ICU stay per patient")
d = d[d["los_h"] >= CFG.min_los_h()].copy();               _estep(d, f"Alive in ICU at anchor + buffer (LOS >= {CFG.min_los_h():.0f} h)")
d = d[d["unitdischargestatus"].notna()].copy();            _estep(d, "Non-missing outcome")

eicu_cohort = d.reset_index(drop=True)
econsort_df = pd.DataFrame(econsort)
econsort_df["excluded"] = econsort_df["stays"].shift(1) - econsort_df["stays"]
econsort_df.to_csv(CFG.TABLE_DIR / "eicu_consort.csv", index=False)
logger.info("eICU final cohort: %d stays", len(eicu_cohort))
econsort_df


21:12:06 | INFO    | START  eICU patient table
21:12:07 | INFO    | DONE   eICU patient table  (1.0s)
21:12:09 | INFO    | eICU final cohort: 128541 stays


,step,stays,patients,excluded
0,All ICU stays,200859,139367,NaN
1,Adults (age >= 18),200234,138868,625.0
2,First ICU stay per patient,138868,138868,61366.0
3,Alive in ICU at anchor + buffer (LOS >= 10 h),128544,128544,10324.0
4,Non-missing outcome,128541,128541,3.0


In [19]:
# --- eICU demographics and outcomes ----------------------------------------
ec = eicu_cohort
ec["sex"] = ec["gender"].map({"Male": "Male", "Female": "Female"}).fillna("Unknown")
ec["race_group"] = ec["ethnicity"].map(EICU_RACE).fillna("Other/Unknown")

_hd = ec["hospitaldischargestatus"].astype(str).str.lower()
_ud = ec["unitdischargestatus"].astype(str).str.lower()
ec["in_hospital_mortality"] = (_hd == "expired").astype(int)
ec["icu_mortality"] = (_ud == "expired").astype(int)

# eICU has no post-discharge death date; 28-day mortality is approximated as an
# in-hospital death occurring within 28 days of unit admission and is left missing
# where hospital discharge timing is unavailable. This is documented as an
# external-validation limitation; in-hospital mortality is the primary outcome.
_hosp_days = pd.to_numeric(ec["hospitaldischargeoffset"], errors="coerce") / 1440.0
ec["hosp_los_days"] = _hosp_days
ec["mortality_28d"] = np.where(
    ec["in_hospital_mortality"].eq(1) & (_hosp_days <= 28), 1.0,
    np.where(ec["in_hospital_mortality"].eq(0), 0.0, np.nan))

ec["admission_type"] = "Unknown"
ec["insurance"] = "Unknown"
ec["patient_id"] = ec["patientunitstayid"].astype("int64")
ec["hadm_id"] = pd.to_numeric(ec["patienthealthsystemstayid"], errors="coerce")

EICU_IDS = ec["patientunitstayid"].astype("int64").tolist()
logger.info("eICU in-hospital mortality: %.1f%% | ICU mortality: %.1f%%",
            100 * ec["in_hospital_mortality"].mean(), 100 * ec["icu_mortality"].mean())
ec[["patient_id", "admission_age", "sex", "race_group", "los_h",
    "in_hospital_mortality", "icu_mortality", "mortality_28d"]].head()


21:12:09 | INFO    | eICU in-hospital mortality: 8.6% | ICU mortality: 5.2%


In [20]:
# --- eICU vitals (vitalPeriodic + vitalAperiodic) + GCS/FiO2 (apacheApsVar) --
if cache_exists("eicu_vitals"):
    eicu_vitals = load_cache("eicu_vitals")
    logger.info("loaded eicu_vitals from cache")
else:
    with timed("eICU vitals extraction"):
        parts = eicu_collect_wide(eicu_path("vitalPeriodic"), EICU_PERIODIC,
                                  EICU_IDS, CFG.CHUNK_LARGE, "vitalPeriodic")
        parts += eicu_collect_wide(eicu_path("vitalAperiodic"), EICU_APERIODIC,
                                   EICU_IDS, CFG.CHUNK_MED, "vitalAperiodic")
        vit_wide = _eicu_finalise_wide(parts, "patientunitstayid")

        # GCS components and FiO2 from the APACHE physiology extract (first day)
        _avcols = ["patientunitstayid", "eyes", "motor", "verbal", "fio2",
                   "vent", "dialysis", "intubated"]
        av = pd.read_csv(eicu_path("apacheApsVar"), usecols=lambda c: c in _avcols,
                         low_memory=False)
        av = av[av["patientunitstayid"].isin(set(EICU_IDS))].copy()
        for c in ["eyes", "motor", "verbal", "fio2"]:
            if c in av.columns:
                av[c] = pd.to_numeric(av[c], errors="coerce")
                av[c] = av[c].where(av[c] >= 0, np.nan)   # eICU uses -1 for missing
        if "fio2" in av.columns:
            av["fio2"] = np.where(av["fio2"] > 1, av["fio2"] / 100.0, av["fio2"])
        gcsfio2 = pd.DataFrame({"patientunitstayid": av["patientunitstayid"].values})
        for src, feat in [("eyes", "gcs_eye"), ("verbal", "gcs_verbal"),
                          ("motor", "gcs_motor"), ("fio2", "fio2")]:
            if src in av.columns:
                for st in STATS:
                    gcsfio2[f"{feat}_{st}"] = av[src].values

        eicu_vitals = vit_wide.merge(gcsfio2, on="patientunitstayid", how="outer")
        eicu_vitals = eicu_vitals.rename(columns={"patientunitstayid": "patient_id"})
        # keep the APACHE support signals for treatment fall-back
        eicu_apache_support = av[["patientunitstayid"] +
                                 [c for c in ["vent", "dialysis", "intubated"] if c in av.columns]].copy()
        save_cache(eicu_vitals, "eicu_vitals")
        eicu_apache_support.to_parquet(CFG.CACHE_DIR / "eicu_apache_support.parquet", index=False)
        del parts, vit_wide; gc.collect()

if "eicu_apache_support" not in dir():
    _p = CFG.CACHE_DIR / "eicu_apache_support.parquet"
    eicu_apache_support = pd.read_parquet(_p) if _p.exists() else pd.DataFrame(columns=["patientunitstayid"])
logger.info("eICU vitals wide table: %d stays x %d columns", *eicu_vitals.shape)
eicu_vitals.head(3)


21:12:09 | INFO    | loaded eicu_vitals from cache
21:12:09 | INFO    | eICU vitals wide table: 128106 stays x 49 columns


In [21]:
# --- eICU labs (lab table, single pass) ------------------------------------
if cache_exists("eicu_labs"):
    eicu_labs = load_cache("eicu_labs")
    logger.info("loaded eicu_labs from cache")
else:
    with timed("eICU lab extraction"):
        parts = eicu_collect_long(eicu_path("lab"),
                                  id_col="patientunitstayid", off_col="labresultoffset",
                                  name_col="labname", val_col="labresult",
                                  name_map=EICU_LABS, ids=EICU_IDS,
                                  chunksize=CFG.CHUNK_LARGE, label="lab")
        eicu_labs = _eicu_finalise_wide(parts, "patientunitstayid")
        eicu_labs = eicu_labs.rename(columns={"patientunitstayid": "patient_id"})
        save_cache(eicu_labs, "eicu_labs")
        del parts; gc.collect()

logger.info("eICU labs wide table: %d stays x %d columns", *eicu_labs.shape)
eicu_labs.head(3)


21:12:09 | INFO    | loaded eicu_labs from cache
21:12:09 | INFO    | eICU labs wide table: 98563 stays x 81 columns


In [22]:
# --- eICU post-anchor treatments (treatment + infusionDrug, string parsed) --
def _eicu_treatment_flags(path, id_col, off_col, str_col, ids, chunksize, label):
    """Per-chunk boolean flags for treatment categories and vasoactive agents,
    split into post-anchor and day-1 timing. Returns a list of per-id boolean
    aggregates to be OR-combined across chunks and sources."""
    idset = set(ids)
    parts = []
    reader = pd.read_csv(path, usecols=lambda c: c in (id_col, off_col, str_col),
                         chunksize=chunksize, low_memory=False)
    for chunk in tqdm(reader, desc=f"scan {label}", unit="chunk"):
        chunk = chunk[chunk[id_col].isin(idset)]
        if chunk.empty:
            continue
        off = pd.to_numeric(chunk[off_col], errors="coerce")
        s = chunk[str_col].astype(str).str.lower().fillna("")
        post = off > EICU_ANCHOR_MIN
        day1 = (off >= 0) & (off <= EICU_ANCHOR_MIN)
        vaso = pd.Series(False, index=chunk.index)
        agent_hit = {}
        for agent, kws in EICU_VASO_AGENTS.items():
            hit = s.str.contains("|".join(kws), regex=True)
            agent_hit[agent] = hit
            vaso = vaso | hit
        for g in EICU_VASO_GENERIC:
            vaso = vaso | s.str.contains(g, regex=False)
        ino = s.str.contains("|".join(EICU_INOTROPE_KW), regex=True)
        vent = s.str.contains("|".join(EICU_VENT_KW), regex=True)
        invvent = s.str.contains("|".join(EICU_INVVENT_KW), regex=True)
        rrt = s.str.contains("|".join(EICU_RRT_KW), regex=True)
        tmp = pd.DataFrame({id_col: chunk[id_col].values,
                            "vaso_post": (vaso & post).values,
                            "vaso_day1": (vaso & day1).values,
                            "inotrope_post": (ino & post).values,
                            "vent_post": (vent & post).values,
                            "vent_day1": (vent & day1).values,
                            "invasive_vent_post": (invvent & post).values,
                            "rrt_post": (rrt & post).values,
                            "rrt_day1": (rrt & day1).values})
        for agent, hit in agent_hit.items():
            tmp[f"ag_{agent}_post"] = (hit & post).values
        parts.append(tmp.groupby(id_col).max())
    return parts

if cache_exists("eicu_treat"):
    eicu_treat = load_cache("eicu_treat")
    logger.info("loaded eicu_treat from cache")
else:
    with timed("eICU treatment extraction"):
        parts = _eicu_treatment_flags(eicu_path("treatment"), "patientunitstayid",
                                      "treatmentoffset", "treatmentstring",
                                      EICU_IDS, CFG.CHUNK_MED, "treatment")
        # infusionDrug contributes vasoactive-agent evidence via drugname
        try:
            parts += _eicu_treatment_flags(eicu_path("infusionDrug"), "patientunitstayid",
                                           "infusionoffset", "drugname",
                                           EICU_IDS, CFG.CHUNK_MED, "infusionDrug")
        except Exception as e:
            logger.warning("infusionDrug parse skipped: %s", e)

        base = pd.DataFrame({"patientunitstayid": pd.Series(EICU_IDS, dtype="int64")}).set_index("patientunitstayid")
        if parts:
            comb = pd.concat(parts).groupby(level=0).max()
            flags = base.join(comb, how="left").fillna(False)
        else:
            flags = base
        agent_cols = [c for c in flags.columns if c.startswith("ag_")]
        for c in ["vaso_post", "vaso_day1", "inotrope_post", "vent_post", "vent_day1",
                  "invasive_vent_post", "rrt_post", "rrt_day1"] + agent_cols:
            if c not in flags.columns:
                flags[c] = False

        # APACHE fall-back for first-day ventilation / dialysis (string tables miss some)
        if len(eicu_apache_support):
            sup = eicu_apache_support.set_index("patientunitstayid")
            if "vent" in sup.columns:
                vflag = pd.to_numeric(sup["vent"], errors="coerce").reindex(flags.index) == 1
                flags["vent_day1"] = flags["vent_day1"] | vflag.fillna(False).values
                flags["vent_post"] = flags["vent_post"] | vflag.fillna(False).values
            if "intubated" in sup.columns:
                iflag = pd.to_numeric(sup["intubated"], errors="coerce").reindex(flags.index) == 1
                flags["invasive_vent_post"] = flags["invasive_vent_post"] | iflag.fillna(False).values
            if "dialysis" in sup.columns:
                dflag = pd.to_numeric(sup["dialysis"], errors="coerce").reindex(flags.index) == 1
                flags["rrt_day1"] = flags["rrt_day1"] | dflag.fillna(False).values
                flags["rrt_post"] = flags["rrt_post"] | dflag.fillna(False).values

        flags["n_pressor_types_post"] = flags[agent_cols].sum(axis=1) if agent_cols else 0
        flags["any_organ_support_post"] = (flags["vaso_post"] | flags["vent_post"] | flags["rrt_post"])

        out = flags.reset_index().rename(columns={"patientunitstayid": "patient_id"})
        for c in ["vaso_post", "vaso_day1", "inotrope_post", "vent_post", "vent_day1",
                  "invasive_vent_post", "rrt_post", "rrt_day1", "any_organ_support_post"]:
            out[c] = out[c].astype("int8")
        out["n_pressor_types_post"] = pd.to_numeric(out["n_pressor_types_post"], errors="coerce").fillna(0).astype("int8")
        eicu_treat = out[["patient_id", "vaso_post", "vaso_day1", "inotrope_post",
                          "vent_post", "vent_day1", "invasive_vent_post",
                          "rrt_post", "rrt_day1", "n_pressor_types_post",
                          "any_organ_support_post"]]
        save_cache(eicu_treat, "eicu_treat")
        del parts, flags; gc.collect()

logger.info("eICU post-anchor treatment rates: vaso %.1f%% | vent %.1f%% | rrt %.1f%%",
            100 * eicu_treat.vaso_post.mean(), 100 * eicu_treat.vent_post.mean(),
            100 * eicu_treat.rrt_post.mean())
eicu_treat.head(3)


21:12:09 | INFO    | loaded eicu_treat from cache
21:12:09 | INFO    | eICU post-anchor treatment rates: vaso 16.3% | vent 33.8% | rrt 4.5%


In [23]:
# --- eICU comorbidities (Elixhauser from diagnosis.icd9code) ----------------
if cache_exists("eicu_comorbid"):
    eicu_comorbid = load_cache("eicu_comorbid")
    logger.info("loaded eicu_comorbid from cache")
else:
    with timed("eICU comorbidities"):
        dg = pd.read_csv(eicu_path("diagnosis"),
                         usecols=lambda c: c in ("patientunitstayid", "icd9code"),
                         low_memory=False)
        dg = dg[dg["patientunitstayid"].isin(set(EICU_IDS))].dropna(subset=["icd9code"]).copy()
        dg["code"] = dg["icd9code"].astype(str).str.split(",")
        dg = dg.explode("code")
        dg["code"] = dg["code"].str.strip()
        dg = dg[dg["code"] != ""]
        dg["ver"] = np.where(dg["code"].str[:1].str.isalpha(), "10", "9")
        uniq = dg[["code", "ver"]].drop_duplicates()
        code2cats = {(r.code, r.ver): elix_categories_for(r.code, r.ver)
                     for r in uniq.itertuples(index=False)}
        dg["cats"] = [code2cats.get((c, v), set()) for c, v in zip(dg["code"], dg["ver"])]
        ex = dg.explode("cats").dropna(subset=["cats"])
        if len(ex):
            flags = (ex.assign(v=1)
                       .pivot_table(index="patientunitstayid", columns="cats",
                                    values="v", aggfunc="max", fill_value=0))
        else:
            flags = pd.DataFrame(index=pd.Index([], name="patientunitstayid"))
        for cat in ELIX:
            if cat not in flags.columns:
                flags[cat] = 0
        flags = flags[list(ELIX.keys())].astype("int8")
        flags["elix_count"] = flags.sum(axis=1).astype("int16")
        flags["van_walraven"] = sum(flags[cat] * w for cat, w in VW_WEIGHTS.items()).astype("int16")
        eicu_comorbid = flags.reset_index().rename(columns={"patientunitstayid": "patient_id"})
        save_cache(eicu_comorbid, "eicu_comorbid")

logger.info("eICU comorbidity table: %d stays, median Elixhauser count = %.0f",
            len(eicu_comorbid), eicu_comorbid["elix_count"].median() if len(eicu_comorbid) else 0)
eicu_comorbid.head(3)


21:12:09 | INFO    | loaded eicu_comorbid from cache
21:12:09 | INFO    | eICU comorbidity table: 68666 stays, median Elixhauser count = 1


In [24]:
# --- eICU first-day SOFA (reuses the MIMIC scoring function) ----------------
# compute_sofa keys on a 'stay_id' column; alias patient_id so the identical
# scoring logic applies to the harmonised eICU physiology.
_ev = eicu_vitals.rename(columns={"patient_id": "stay_id"})
_el = eicu_labs.rename(columns={"patient_id": "stay_id"})
_et = eicu_treat.rename(columns={"patient_id": "stay_id"})
eicu_sofa = compute_sofa(_ev, _el, _et).rename(columns={"stay_id": "patient_id"})
logger.info("eICU first-day SOFA: median %.0f (IQR %.0f-%.0f), mean coverage %.2f",
            eicu_sofa.sofa_total.median(),
            eicu_sofa.sofa_total.quantile(.25), eicu_sofa.sofa_total.quantile(.75),
            eicu_sofa.sofa_coverage.mean())
del _ev, _el, _et; gc.collect()
eicu_sofa.head(3)


21:12:10 | INFO    | eICU first-day SOFA: median 2 (IQR 1-4), mean coverage 0.51


In [25]:
# --- Assemble the eICU analytic table (same canonical schema) ---------------
_edemo = ["patient_id", "hadm_id", "admission_age", "sex", "race_group", "los_h",
          "admission_type", "insurance", "in_hospital_mortality", "icu_mortality",
          "mortality_28d", "hosp_los_days"]
eicu_asm = ec[_edemo].copy()
eicu_asm = eicu_asm.merge(eicu_comorbid, on="patient_id", how="left")
for _tbl in (eicu_vitals, eicu_labs, eicu_treat, eicu_sofa):
    eicu_asm = eicu_asm.merge(_tbl, on="patient_id", how="left")

eicu_analytic = finalise_schema(eicu_asm, "eICU")
eicu_analytic = downcast(eicu_analytic)
save_cache(eicu_analytic, "eicu_analytic")
eicu_analytic.to_parquet(CFG.OUTPUT_DIR / "eicu_analytic.parquet", index=False)
logger.info("eICU analytic table: %d rows x %d cols", *eicu_analytic.shape)
eicu_analytic.head(3)


21:12:16 | INFO    | cached eicu_analytic                rows=128541   cols=229
21:12:19 | INFO    | eICU analytic table: 128541 rows x 229 cols


In [26]:
# --- Feature dictionary: role, dtype, per-database missingness (Supplementary Table S1) ---
role_map = {}
for c in ID_COLUMNS:       role_map[c] = "identifier"
for c in DEMO_COLUMNS:     role_map[c] = "demographic"
for c in OUTCOME_COLUMNS:  role_map[c] = "outcome"
for c in TREAT_COLUMNS:    role_map[c] = "treatment (post-anchor / exposure)"
for c in SOFA_COLUMNS:     role_map[c] = "severity (first-day SOFA)"
for c in COMORBID_COLUMNS: role_map[c] = "comorbidity"
for c in PHYSIO_COLUMNS:   role_map[c] = "physiology (pre-anchor)"

rows = [{"feature": c,
         "role": role_map.get(c, "other"),
         "dtype": str(mimic_analytic[c].dtype),
         "mimic_missing_pct": round(100 * mimic_analytic[c].isna().mean(), 1),
         "eicu_missing_pct": round(100 * eicu_analytic[c].isna().mean(), 1) if c in eicu_analytic.columns else None}
        for c in mimic_analytic.columns]
feature_dictionary = pd.DataFrame(rows)
feature_dictionary.to_csv(CFG.TABLE_DIR / "feature_dictionary.csv", index=False)
logger.info("Feature dictionary written: %d columns", len(feature_dictionary))
feature_dictionary.head(10)

21:12:19 | INFO    | Feature dictionary written: 229 columns


,feature,role,dtype,mimic_missing_pct,eicu_missing_pct
0,db,identifier,object,0.0,0.0
1,patient_id,identifier,int32,0.0,0.0
2,hadm_id,identifier,int32,0.0,0.0
3,admission_age,demographic,int8,0.0,0.0
4,sex,demographic,object,0.0,0.0
5,race_group,demographic,object,0.0,0.0
6,los_h,demographic,float32,0.0,0.0
7,admission_type,demographic,object,0.0,0.0
8,insurance,demographic,object,0.0,0.0
9,in_hospital_mortality,outcome,int32,0.0,0.0


---
## Part C. Quality control, missingness, baseline characteristics, exploratory analysis

The two analytic tables are now schema-identical. This part characterises physiologic plausibility, quantifies missingness per feature and per database, builds the CONSORT flow diagram, produces the baseline characteristics table stratified by database and by in-hospital mortality, and generates the exploratory figures a reviewer expects. Every figure is written to disk as 300-dpi PNG and vector PDF.

In [27]:
# --- Physiologic plausibility clamps (documented, conservative) -------------
# Values outside these ranges are set to missing rather than winsorised, so no
# implausible reading enters the summaries. Ranges are wide on purpose; they
# remove recording errors, not clinical extremes.
PLAUSIBLE = {
    "heart_rate": (10, 300), "sbp": (20, 300), "dbp": (5, 225), "mbp": (10, 250),
    "resp_rate": (2, 90), "temp": (25, 45), "spo2": (20, 100), "fio2": (0.20, 1.0),
    "gcs_eye": (1, 4), "gcs_verbal": (1, 5), "gcs_motor": (1, 6),
    "weight_kg": (20, 400), "height_cm": (100, 250),
    "creatinine": (0.1, 25), "bun": (1, 250), "sodium": (90, 200), "potassium": (1, 12),
    "bicarbonate": (2, 60), "chloride": (60, 160), "glucose": (10, 2000),
    "hemoglobin": (2, 25), "hematocrit": (5, 75), "wbc": (0.01, 500),
    "platelets": (1, 2000), "lactate": (0.1, 40), "ph": (6.5, 8.0),
    "pao2": (20, 700), "paco2": (5, 200), "bilirubin_total": (0.1, 80),
    "albumin": (0.5, 7), "inr": (0.5, 20), "ptt": (10, 250), "anion_gap": (1, 60),
}
def apply_plausibility(df):
    df = df.copy()
    n_removed = 0
    for feat, (lo, hi) in PLAUSIBLE.items():
        for st in STATS:
            col = f"{feat}_{st}"
            if col in df.columns:
                bad = (df[col] < lo) | (df[col] > hi)
                n_removed += int(bad.sum())
                df.loc[bad, col] = np.nan
    return df, n_removed

mimic_analytic, n_m = apply_plausibility(mimic_analytic)
eicu_analytic,  n_e = apply_plausibility(eicu_analytic)
logger.info("Plausibility filtering removed %d MIMIC and %d eICU implausible cell values", n_m, n_e)

# refresh the on-disk analytic tables after cleaning
mimic_analytic.to_parquet(CFG.OUTPUT_DIR / "mimic_analytic.parquet", index=False)
eicu_analytic.to_parquet(CFG.OUTPUT_DIR / "eicu_analytic.parquet", index=False)

combined = pd.concat([mimic_analytic, eicu_analytic], ignore_index=True)
combined.to_parquet(CFG.OUTPUT_DIR / "analytic_combined.parquet", index=False)
logger.info("Combined analytic table: %d rows (MIMIC %d + eICU %d)",
            len(combined), len(mimic_analytic), len(eicu_analytic))
combined.groupby("db")[["in_hospital_mortality", "icu_mortality", "any_organ_support_post"]].mean().round(3)


21:12:20 | INFO    | Plausibility filtering removed 1014650 MIMIC and 15532 eICU implausible cell values
21:12:26 | INFO    | Combined analytic table: 178225 rows (MIMIC 49684 + eICU 128541)


,in_hospital_mortality,icu_mortality,any_organ_support_post
db,,,
MIMIC,0.098,0.065,0.456
eICU,0.086,0.052,0.408


In [28]:
# --- Missingness analysis (per feature, per database) ----------------------
FEATURE_COLUMNS = PHYSIO_COLUMNS + SOFA_COLUMNS + TREAT_COLUMNS + COMORBID_COLUMNS

def missingness_table(df):
    miss = df[FEATURE_COLUMNS].isna().mean().rename("missing_fraction").reset_index()
    miss = miss.rename(columns={"index": "feature"})
    return miss.sort_values("missing_fraction", ascending=False)

miss_m = missingness_table(mimic_analytic).rename(columns={"missing_fraction": "MIMIC"})
miss_e = missingness_table(eicu_analytic).rename(columns={"missing_fraction": "eICU"})
missing = miss_m.merge(miss_e, on="feature", how="outer")
missing.to_csv(CFG.TABLE_DIR / "missingness_by_database.csv", index=False)

# heatmap of the most-incomplete physiology features (aggregate stats only)
core_feats = [f"{f}_mean" for f in (VITAL_FEATURES + LAB_FEATURES)]
hm = missing.set_index("feature").reindex(core_feats)[["MIMIC", "eICU"]]
hm = hm.dropna(how="all")
fig, ax = plt.subplots(figsize=(6, max(4, 0.28 * len(hm))))
im = ax.imshow(hm.values, aspect="auto", cmap="magma", vmin=0, vmax=1)
ax.set_xticks([0, 1]); ax.set_xticklabels(["MIMIC-IV", "eICU"])
ax.set_yticks(range(len(hm))); ax.set_yticklabels([c.replace("_mean", "") for c in hm.index], fontsize=7)
ax.set_title("Feature missingness (mean-statistic columns)")
cbar = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04); cbar.set_label("fraction missing")
for (i, j), v in np.ndenumerate(hm.values):
    if np.isfinite(v):
        ax.text(j, i, f"{v:.2f}", ha="center", va="center",
                color="white" if v > 0.5 else "black", fontsize=6)
savefig(fig, "fig_missingness_heatmap")
missing.head(12)


21:12:30 | INFO    | figure -> fig_missingness_heatmap (.png/.pdf)


,feature,MIMIC,eICU
0,albumin_last,0.836144,0.804164
1,albumin_max,0.836144,0.804164
2,albumin_mean,0.836144,0.804164
3,albumin_min,0.836144,0.804164
4,albumin_slope,0.999074,1.000000
5,alcohol,0.000000,0.000000
6,anion_gap_last,0.309315,0.661649
7,anion_gap_max,0.309315,0.661633
8,anion_gap_mean,0.309295,0.661633
9,anion_gap_min,0.309355,0.661672


In [29]:
# =============================================================================
# P11: MICE imputation sensitivity
# =============================================================================
try:
    from sklearn.experimental import enable_iterative_imputer
    from sklearn.impute import IterativeImputer
    from sklearn.preprocessing import StandardScaler
    
    logger.info("Running MICE imputation sensitivity...")
    # Select a subset of features for speed
    # Define numeric columns from the analytic table
    numeric_cols = mimic_analytic.select_dtypes(include=[np.number]).columns.tolist()
    mice_feats = [c for c in numeric_cols if c in mimic_analytic.columns][:30]
    X_subset = mimic_analytic[mice_feats].copy()
    
    # Standardize
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X_subset)
    
    # MICE imputation
    mice = IterativeImputer(max_iter=10, random_state=CFG.RANDOM_SEED)
    X_mice = mice.fit_transform(X_scaled)
    X_mice_df = pd.DataFrame(X_mice, columns=mice_feats)
    
    # Compare to median imputation
    from sklearn.impute import SimpleImputer
    median_imp = SimpleImputer(strategy='median')
    X_median = median_imp.fit_transform(X_subset)
    X_median_df = pd.DataFrame(X_median, columns=mice_feats)
    
    # Compute correlation between methods
    corr_df = pd.DataFrame({
        "feature": mice_feats,
        "correlation_mice_median": [float(np.corrcoef(X_mice_df[c], X_median_df[c])[0,1]) for c in mice_feats]
    })
    corr_df.to_csv(CFG.TABLE_DIR / "mice_sensitivity.csv", index=False)
    logger.info("MICE sensitivity complete. Mean correlation: %.3f", corr_df["correlation_mice_median"].mean())
except Exception as e:
    logger.warning("MICE sensitivity skipped: %s", e)

21:12:30 | INFO    | Running MICE imputation sensitivity...
21:13:06 | INFO    | MICE sensitivity complete. Mean correlation: 0.910


In [30]:
# =============================================================================
# P11: Little's MCAR test for missingness mechanism
# =============================================================================
try:
    from scipy.stats import chi2
    # Quick MCAR test: compare mean of missing vs observed for key features
    mcar_results = []
    for feat in VITAL_FEATURES + LAB_FEATURES[:10]:  # subset for speed
        col = f"{feat}_mean"
        if col in mimic_analytic.columns:
            # Test if missingness is correlated with observed values
            obs = mimic_analytic[col].dropna()
            mis = mimic_analytic[col].isna()
            if len(obs) > 100 and mis.sum() > 100:
                # Compare distribution of other features by missingness
                from scipy.stats import ttest_ind
                other = "lactate_mean" if feat != "lactate" else "creatinine_mean"
                if other in mimic_analytic.columns:
                    t, p = ttest_ind(mimic_analytic[other].values[mis.values], 
                                     mimic_analytic[other].values[~mis.values])
                    mcar_results.append({"feature": col, "p_mcar": round(p, 4)})
    mcar_df = pd.DataFrame(mcar_results)
    mcar_df.to_csv(CFG.TABLE_DIR / "mcar_tests.csv", index=False)
    logger.info("MCAR tests complete for %d features", len(mcar_df))
except Exception as e:
    logger.warning("MCAR test skipped: %s", e)

21:13:06 | INFO    | MCAR tests complete for 23 features


In [31]:
# --- CONSORT flow diagram (both databases) ---------------------------------
def draw_consort(ax, steps_df, title, box_color):
    ax.set_xlim(0, 10); ax.set_ylim(0, 10); ax.axis("off"); ax.set_title(title, fontsize=11, weight="bold")
    n = len(steps_df)
    y_positions = np.linspace(9.2, 0.8, n)
    box_w, box_h = 6.6, 1.15
    for i, (_, row) in enumerate(steps_df.iterrows()):
        y = y_positions[i]
        ax.add_patch(plt.Rectangle((1.7, y - box_h / 2), box_w, box_h,
                                   facecolor=box_color, edgecolor="black", lw=1.1))
        ax.text(1.7 + box_w / 2, y, f"{row['step']}\nn = {int(row['stays']):,} stays",
                ha="center", va="center", fontsize=8)
        if i < n - 1:
            ax.annotate("", xy=(1.7 + box_w / 2, y_positions[i + 1] + box_h / 2),
                        xytext=(1.7 + box_w / 2, y - box_h / 2),
                        arrowprops=dict(arrowstyle="-|>", lw=1.1, color="black"))
            exc = row.get("excluded", np.nan)
            nxt_exc = steps_df.iloc[i + 1].get("excluded", np.nan)
            if pd.notna(nxt_exc) and nxt_exc > 0:
                ymid = (y - box_h / 2 + y_positions[i + 1] + box_h / 2) / 2
                ax.add_patch(plt.Rectangle((8.5, ymid - 0.45, ), 1.2, 0.9,
                                           facecolor="whitesmoke", edgecolor="grey", lw=0.8))
                ax.text(9.1, ymid, f"-{int(nxt_exc):,}", ha="center", va="center", fontsize=7, color="firebrick")

fig, axes = plt.subplots(1, 2, figsize=(13, 7))
draw_consort(axes[0], consort_df, "MIMIC-IV v2.2 (development)", "#cfe3f5")
draw_consort(axes[1], econsort_df, "eICU-CRD v2.0 (external validation)", "#d8f0d8")
fig.suptitle("Cohort construction and landmark eligibility", fontsize=13, weight="bold")
savefig(fig, "fig_consort_flow")
print("CONSORT diagram written.")


21:13:08 | INFO    | figure -> fig_consort_flow (.png/.pdf)
CONSORT diagram written.


In [32]:
# --- Table 1: baseline characteristics, stratified by database and mortality --
def _fmt_cont(s):
    s = pd.to_numeric(s, errors="coerce")
    if s.notna().sum() == 0:
        return "-"
    return f"{np.nanmedian(s):.1f} [{np.nanpercentile(s, 25):.1f}, {np.nanpercentile(s, 75):.1f}]"

def _fmt_bin(s):
    s = pd.to_numeric(s, errors="coerce")
    if s.notna().sum() == 0:
        return "-"
    return f"{int(np.nansum(s)):,} ({100 * np.nanmean(s):.1f}%)"

CONT_VARS = [("admission_age", "Age, years"), ("los_h", "ICU length of stay, h"),
             ("sofa_total", "First-day SOFA"), ("elix_count", "Elixhauser count"),
             ("van_walraven", "van Walraven index"),
             ("heart_rate_mean", "Heart rate, mean"), ("mbp_mean", "Mean arterial pressure"),
             ("resp_rate_mean", "Respiratory rate, mean"), ("spo2_mean", "SpO2, mean"),
             ("creatinine_max", "Creatinine, max"), ("lactate_max", "Lactate, max"),
             ("wbc_max", "WBC, max"), ("platelets_min", "Platelets, min")]
BIN_VARS = [("in_hospital_mortality", "In-hospital mortality"),
            ("icu_mortality", "ICU mortality"),
            ("vaso_post", "Post-anchor vasopressor"),
            ("vent_post", "Post-anchor ventilation"),
            ("rrt_post", "Post-anchor renal replacement"),
            ("any_organ_support_post", "Any post-anchor organ support")]

def table1(df):
    rows = []
    rows.append(("N (stays)", f"{len(df):,}", "", ""))
    surv = df[df["in_hospital_mortality"] == 0]
    died = df[df["in_hospital_mortality"] == 1]
    for col, label in CONT_VARS:
        rows.append((label, _fmt_cont(df[col]), _fmt_cont(surv[col]), _fmt_cont(died[col])))
    for grp in ["Male", "Female"]:
        rows.append((f"Sex: {grp}", _fmt_bin((df["sex"] == grp)),
                     _fmt_bin((surv["sex"] == grp)), _fmt_bin((died["sex"] == grp))))
    for grp in ["White", "Black", "Hispanic", "Asian", "Other/Unknown"]:
        rows.append((f"Race: {grp}", _fmt_bin((df["race_group"] == grp)),
                     _fmt_bin((surv["race_group"] == grp)), _fmt_bin((died["race_group"] == grp))))
    for col, label in BIN_VARS:
        rows.append((label, _fmt_bin(df[col]), _fmt_bin(surv[col]), _fmt_bin(died[col])))
    return pd.DataFrame(rows, columns=["Characteristic", "Overall", "Survived", "Died"])

t1_mimic = table1(mimic_analytic); t1_mimic.insert(0, "Database", "MIMIC-IV")
t1_eicu  = table1(eicu_analytic);  t1_eicu.insert(0, "Database", "eICU")
table1_all = pd.concat([t1_mimic, t1_eicu], ignore_index=True)
table1_all.to_csv(CFG.TABLE_DIR / "table1_baseline_characteristics.csv", index=False)
logger.info("Table 1 written (%d rows).", len(table1_all))
t1_mimic


21:13:09 | INFO    | Table 1 written (54 rows).


,Database,Characteristic,Overall,Survived,Died
0,MIMIC-IV,N (stays),"49,684",,
1,MIMIC-IV,"Age, years","66.0 [54.0, 78.0]","66.0 [53.0, 77.0]","73.0 [61.0, 83.0]"
2,MIMIC-IV,"ICU length of stay, h","46.0 [26.8, 86.9]","44.8 [26.6, 79.9]","69.2 [32.4, 154.0]"
3,MIMIC-IV,First-day SOFA,"3.0 [1.0, 7.0]","3.0 [1.0, 6.0]","6.0 [3.0, 10.0]"
4,MIMIC-IV,Elixhauser count,"4.0 [2.0, 6.0]","4.0 [2.0, 5.0]","5.0 [4.0, 7.0]"
5,MIMIC-IV,van Walraven index,"10.0 [3.0, 17.0]","8.0 [3.0, 16.0]","18.0 [11.0, 25.0]"
6,MIMIC-IV,"Heart rate, mean","82.8 [72.7, 95.9]","82.2 [72.4, 95.0]","89.9 [76.2, 105.2]"
7,MIMIC-IV,Mean arterial pressure,"79.2 [71.5, 88.3]","79.5 [72.0, 88.6]","75.4 [67.4, 85.6]"
8,MIMIC-IV,"Respiratory rate, mean","18.0 [15.8, 21.1]","17.9 [15.7, 20.7]","20.6 [17.5, 24.3]"
9,MIMIC-IV,"SpO2, mean","97.6 [95.8, 99.2]","97.6 [95.9, 99.2]","97.2 [95.0, 99.1]"


In [33]:
# --- eICU Table 1 view ------------------------------------------------------
t1_eicu


,Database,Characteristic,Overall,Survived,Died
0,eICU,N (stays),"128,541",,
1,eICU,"Age, years","65.0 [53.0, 76.0]","65.0 [52.0, 76.0]","72.0 [61.0, 81.0]"
2,eICU,"ICU length of stay, h","42.8 [24.0, 75.8]","41.8 [23.8, 72.2]","65.3 [30.5, 135.3]"
3,eICU,First-day SOFA,"2.0 [1.0, 4.0]","1.0 [1.0, 4.0]","5.0 [3.0, 8.0]"
4,eICU,Elixhauser count,"1.0 [0.0, 1.0]","1.0 [0.0, 1.0]","1.0 [0.0, 2.0]"
5,eICU,van Walraven index,"0.0 [0.0, 5.0]","0.0 [0.0, 5.0]","5.0 [0.0, 10.0]"
6,eICU,"Heart rate, mean","83.8 [72.1, 97.2]","83.1 [71.7, 96.2]","92.1 [77.8, 106.8]"
7,eICU,Mean arterial pressure,"80.8 [71.5, 91.9]","81.3 [72.1, 92.4]","74.8 [66.6, 85.7]"
8,eICU,"Respiratory rate, mean","18.4 [15.8, 22.0]","18.3 [15.7, 21.7]","21.0 [17.3, 25.4]"
9,eICU,"SpO2, mean","97.4 [95.7, 98.9]","97.4 [95.7, 98.9]","97.2 [94.8, 99.0]"


In [34]:
# --- Exploratory figures: outcome and treatment rates by database ----------
rates = (combined.groupby("db")[["in_hospital_mortality", "icu_mortality",
                                 "vaso_post", "vent_post", "rrt_post",
                                 "any_organ_support_post"]].mean() * 100)
fig, ax = plt.subplots(figsize=(9, 4.6))
rates.T.plot(kind="bar", ax=ax, color=["#3b76af", "#5aa469"], edgecolor="black", lw=0.6)
ax.set_ylabel("percent of stays"); ax.set_xlabel("")
ax.set_title("Outcome and post-anchor treatment rates by database")
ax.set_xticklabels(["In-hosp mortality", "ICU mortality", "Vasopressor",
                    "Ventilation", "RRT", "Any organ support"], rotation=25, ha="right")
ax.legend(title="database")
savefig(fig, "fig_outcome_treatment_rates")
rates.round(1)


21:13:10 | INFO    | figure -> fig_outcome_treatment_rates (.png/.pdf)


,in_hospital_mortality,icu_mortality,vaso_post,vent_post,rrt_post,any_organ_support_post
db,,,,,,
MIMIC,9.8,6.5,27.6,38.5,3.7,45.6
eICU,8.6,5.2,16.3,33.8,4.5,40.8


In [35]:
# --- Exploratory figures: SOFA distribution and organ subscores ------------
fig, axes = plt.subplots(1, 2, figsize=(12, 4.6))
for db, colr in [("MIMIC", "#3b76af"), ("eICU", "#5aa469")]:
    sub = combined[combined.db == db]["sofa_total"].dropna()
    axes[0].hist(sub, bins=range(0, 25), alpha=0.55, label=db, color=colr, density=True)
axes[0].set_xlabel("first-day SOFA"); axes[0].set_ylabel("density")
axes[0].set_title("SOFA distribution by database"); axes[0].legend()

sofa_parts = ["sofa_resp", "sofa_coag", "sofa_liver", "sofa_cardio", "sofa_cns", "sofa_renal"]
mean_parts = combined.groupby("db")[sofa_parts].mean()
mean_parts.T.plot(kind="bar", ax=axes[1], color=["#3b76af", "#5aa469"], edgecolor="black", lw=0.6)
axes[1].set_xticklabels(["resp", "coag", "liver", "cardio", "cns", "renal"], rotation=0)
axes[1].set_ylabel("mean subscore"); axes[1].set_title("Mean SOFA organ subscores")
axes[1].legend(title="database")
savefig(fig, "fig_sofa_distribution")
mean_parts.round(2)


21:13:12 | INFO    | figure -> fig_sofa_distribution (.png/.pdf)


,sofa_resp,sofa_coag,sofa_liver,sofa_cardio,sofa_cns,sofa_renal
db,,,,,,
MIMIC,0.83,0.34,0.15,1.10,1.42,0.39
eICU,0.40,0.17,0.07,0.85,0.82,0.30


In [36]:
# --- Exploratory figures: mortality-stratified physiology (violin) ---------
viol_feats = [("sofa_total", "First-day SOFA"), ("lactate_max", "Lactate, max"),
              ("creatinine_max", "Creatinine, max"), ("mbp_min", "MAP, min"),
              ("resp_rate_max", "Resp rate, max"), ("platelets_min", "Platelets, min")]
fig, axes = plt.subplots(2, 3, figsize=(13, 7))
for ax, (col, label) in zip(axes.ravel(), viol_feats):
    data = [combined.loc[(combined.in_hospital_mortality == g) & combined[col].notna(), col].values
            for g in (0, 1)]
    parts = ax.violinplot(data, showmedians=True, widths=0.85)
    for pc, c in zip(parts["bodies"], ["#8ec7d2", "#e07b7b"]):
        pc.set_facecolor(c); pc.set_alpha(0.75)
    ax.set_xticks([1, 2]); ax.set_xticklabels(["survived", "died"])
    ax.set_title(label, fontsize=10)
fig.suptitle("Feature distributions by in-hospital mortality (pooled)", fontsize=12, weight="bold")
savefig(fig, "fig_feature_violins")
print("Violin panel written.")


21:13:21 | INFO    | figure -> fig_feature_violins (.png/.pdf)
Violin panel written.


In [37]:
# --- Exploratory figures: feature correlation heatmap ----------------------
corr_feats = ["admission_age", "sofa_total", "elix_count", "van_walraven",
              "heart_rate_mean", "mbp_mean", "resp_rate_mean", "spo2_mean", "temp_mean",
              "creatinine_max", "bun_max", "lactate_max", "wbc_max", "platelets_min",
              "bicarbonate_min", "hemoglobin_min", "ph_min", "pao2_min"]
corr_feats = [c for c in corr_feats if c in combined.columns]
C = combined[corr_feats].astype(float).corr()
fig, ax = plt.subplots(figsize=(9, 8))
im = ax.imshow(C.values, cmap="RdBu_r", vmin=-1, vmax=1)
ax.set_xticks(range(len(corr_feats))); ax.set_xticklabels(corr_feats, rotation=90, fontsize=7)
ax.set_yticks(range(len(corr_feats))); ax.set_yticklabels(corr_feats, fontsize=7)
ax.set_title("Feature correlation (pooled cohort)")
fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
savefig(fig, "fig_correlation_heatmap")
print("Correlation heatmap written.")


21:13:24 | INFO    | figure -> fig_correlation_heatmap (.png/.pdf)
Correlation heatmap written.


In [38]:
# --- Data-quality summary table --------------------------------------------
def quality_summary(df, db):
    return {
        "database": db,
        "stays": len(df),
        "in_hospital_mortality_%": round(100 * df["in_hospital_mortality"].mean(), 2),
        "median_age": round(df["admission_age"].median(), 1),
        "median_sofa": round(df["sofa_total"].median(), 1),
        "mean_sofa_coverage": round(df["sofa_coverage"].mean(), 3),
        "any_organ_support_%": round(100 * df["any_organ_support_post"].mean(), 2),
        "median_feature_missing_%": round(100 * df[FEATURE_COLUMNS].isna().mean().median(), 2),
    }
quality = pd.DataFrame([quality_summary(mimic_analytic, "MIMIC-IV"),
                        quality_summary(eicu_analytic, "eICU")])
quality.to_csv(CFG.TABLE_DIR / "data_quality_summary.csv", index=False)
quality


,database,stays,in_hospital_mortality_%,median_age,median_sofa,mean_sofa_coverage,any_organ_support_%,median_feature_missing_%
0,MIMIC-IV,49684,9.79,66.0,3.0,0.655,45.62,27.36
1,eICU,128541,8.60,65.0,2.0,0.515,40.80,57.75


In [39]:
# --- Artefact manifest ------------------------------------------------------
# A written record of every produced artefact and the key parameters, so the
# data foundation is auditable and Notebook 2 can assert its inputs.
RUN_MANIFEST["finished"] = datetime.now().isoformat(timespec="seconds")
RUN_MANIFEST["config"] = {
    "pred_horizon_h": CFG.PRED_HORIZON_H, "post_buffer_h": CFG.POST_BUFFER_H,
    "feature_window_h": CFG.FEATURE_WINDOW_H, "min_age": CFG.MIN_AGE,
    "first_stay_key": CFG.FIRST_STAY_KEY, "seed": CFG.RANDOM_SEED,
}
RUN_MANIFEST["cohorts"] = {
    "mimic_stays": int(len(mimic_analytic)), "eicu_stays": int(len(eicu_analytic)),
}
RUN_MANIFEST["schema"] = {
    "n_columns": len(CANON), "n_physiology": len(PHYSIO_COLUMNS),
    "n_treatment": len(TREAT_COLUMNS), "n_comorbidity": len(COMORBID_COLUMNS),
    "canonical_columns": CANON,
}
RUN_MANIFEST["outputs"] = {
    "mimic_analytic": str((CFG.OUTPUT_DIR / "mimic_analytic.parquet").resolve()),
    "eicu_analytic": str((CFG.OUTPUT_DIR / "eicu_analytic.parquet").resolve()),
    "analytic_combined": str((CFG.OUTPUT_DIR / "analytic_combined.parquet").resolve()),
    "tables_dir": str(CFG.TABLE_DIR.resolve()),
    "figures_dir": str(CFG.FIG_DIR.resolve()),
}
with open(CFG.OUTPUT_DIR / "nb1_manifest.json", "w", encoding="utf-8") as f:
    json.dump(RUN_MANIFEST, f, indent=2)
logger.info("Manifest written to %s", (CFG.OUTPUT_DIR / "nb1_manifest.json").resolve())

print("Notebook 1 complete.")
print(f"  MIMIC-IV analytic stays : {len(mimic_analytic):,}")
print(f"  eICU analytic stays     : {len(eicu_analytic):,}")
print(f"  Canonical columns       : {len(CANON)}")
print(f"  Outputs under           : {CFG.OUTPUT_DIR.resolve()}")
print("Artefacts for Notebook 2: mimic_analytic.parquet, eicu_analytic.parquet, analytic_combined.parquet")


21:13:25 | INFO    | Manifest written to C:\Users\<user>\Downloads\nb1_outputs_adm\nb1_manifest.json
Notebook 1 complete.
  MIMIC-IV analytic stays : 49,684
  eICU analytic stays     : 128,541
  Canonical columns       : 229
  Outputs under           : C:\Users\<user>\Downloads\nb1_outputs_adm
Artefacts for Notebook 2: mimic_analytic.parquet, eicu_analytic.parquet, analytic_combined.parquet


---
### Handover to Notebook 2

Notebook 2 consumes the three Parquet tables written above (`mimic_analytic.parquet`, `eicu_analytic.parquet`, `analytic_combined.parquet`) plus `nb1_manifest.json`. All share the canonical schema fixed in Part A: identifier and demographic columns, four harmonised outcomes, per-feature `min`/`max`/`mean`/`last` physiology, first-day SOFA, post-anchor treatment indicators, and Elixhauser comorbidity flags with the van Walraven index. Because features derive strictly from the pre-anchor window and treatments strictly from the post-anchor window, the landmark geometry required by the prescriptive-safety audit (g-formula, inverse probability weighting, marginal structural models, and counterfactual treatment-policy comparison) is already guaranteed by construction. Notebook 2 develops the prediction models, estimates treatment effects under that geometry, and quantifies when a well-calibrated score would move treatment in a direction that harms patients.